# Magic Hour | AI Voice Editor
## Production-oriented Colab runner · AuK Base / Flash

Upload a voice recording, choose an edit, compare the result, and download a WAV. This replaces the fixed examples in `tencent.ipynb` with a reusable single-worker implementation.

**Start:** Select a **T4 GPU**, run setup through section 4, upload one file in section 5, then set the controls and run section 6. Do not use **Run all**: it would also re-run setup and model downloads. Run the numbered cells one at a time. No generation, benchmark, approval, export, or deletion runs until its checkbox is enabled.

**Included:** immutable model revision; SHA-256 checks; versioned runtime; real Base/Flash profiles; short-clip processing; one job at a time; SQLite status and idempotency; bounded subprocess and HTTP requests; hung-process termination; input/output decoding checks; optional CPU transcription checks; explicit human review; cold/warm benchmarks; private results and diagnostics.

**Scope:** voice editing, not general music/SFX editing. At most **15 seconds of input** per job, **20 seconds of output**, and a conservative **30-second combined budget**. Longer files require an explicit range. The result is the selected clip only, in mono 24 kHz; this notebook does not silently trim a file, splice a segment into a long recording, or remux/lip-sync a video.

**Validation:** 64 local tests passed on CPU, including real local-HTTP fixtures and simulated failures. No actual AuK or Whisper model inference was executed during notebook construction. Run the GPU acceptance checklist at the end before rollout. Quantized/Flash output quality is not established by a successful WAV export.

**Deployment boundary:** this is a hardened, single-user runner and production handoff candidate, not a multi-tenant service or a guarantee that Colab is suitable hosting. Job recovery lasts only while its local disk survives. Production still needs durable object storage/queueing, authentication and asset ownership, usage/billing integration, isolation, retention policy, and GPU acceptance tests.

**License:** Qwen2.5-Omni-3B remains part of this stack. Its published license permits research/evaluation and requires a separate license for commercial use. The commercial switch below stays blocked without explicit clearance. A checkbox does not obtain a license. See the source notes at the end.

**v0.2.5 changes (untested on GPU):** transcript replace now sends the take-home/upstream prompt "Replace 'old' with 'new'." (v0.2.4 sent "Regenerate this speech as: ..."); new `TRANSCRIPT_SCOPE = whole_clip` edits the whole <=15 s utterance, which needs no hand-checked boundaries but does NOT test localized splice seams (use `localized` for that). ASR/speaker QA gates are unchanged and strict. P0 now restores only corpus/runs from the old backup and writes to a new backup folder; the old backup is never modified.

**Next (live runtime):** refresh this tab without saving an old copy. P1 -> P0 (leave RESTORE_CHECKPOINT unticked; P1 clears the old checkpoint handle so nothing syncs into the old backup) -> P4 (must be rerun after P1 so the new backend is loaded; model files already on disk should be reused, unverified) -> single P7 tests. **Next (recycled runtime):** choose a T4, P1 -> tick RESTORE_CHECKPOINT in P0 -> P2 -> P3 -> P4 -> P5. P2/P4 download models. Setup downloads and GPU allocation may cost money; do not start a paid runtime without a budget.


In [ ]:
# @title P1. Install tools and restore reusable backend v0.3.0 transplant candidate (fixed-duration scope fix; no inference)
# v0.3.0 (2026-10-18): localized edits now enforce the fixed-duration verified-interval contract:
# the cut (changed words + join pads) is clamped strictly inside [segment_start, segment_end] and never
# clips neighbor words; the donor patch must fit the cut frame-exact (room-tone flex only, no stretch,
# no tail shift); otherwise it fails closed 'transplant_duration_mismatch' (widen the interval or use
# whole_clip scope). Supersedes v0.2.9-locfix, whose LOC candidate violated the preservation contract
# (cut extended 8.4ms before the verified interval [1.34,2.2], clipped neighbor word PCM, shifted the
# tail +0.32s). 175/175 CPU tests. GPU L4 retest 2026-10-18: LOC case fails closed on 3/3 seeds
# (donor 'stronger' 0.38s > max contract-clean cut 0.22s; contiguous neighbors) - honest limitation,
# not an impossibility proof for duration-steered strategies.
import sys, subprocess, pathlib, base64, hashlib, zipfile, io, os
if not (3,11) <= sys.version_info[:2] <= (3,13):
    raise RuntimeError("Use a Python 3.11-3.13 Colab runtime.")
WORK_DIR = "/content/auk-production"  # @param {type:"string"}
COMMERCIAL_USE = False  # @param {type:"boolean"}
COMMERCIAL_LICENSE_CONFIRMED = False  # @param {type:"boolean"}
if COMMERCIAL_USE and not COMMERCIAL_LICENSE_CONFIRMED: raise RuntimeError("Obtain commercial model-license clearance first.")
ROOT=pathlib.Path(WORK_DIR).resolve()
if str(ROOT) in ("/", "/content", "/tmp", str(pathlib.Path.home())): raise ValueError("Use a dedicated working directory")
ROOT.mkdir(parents=True,exist_ok=True);os.umask(0o077)
subprocess.run(["apt-get","-qq","update"],check=True)
subprocess.run(["apt-get","-qq","install","-y","ca-certificates","curl","ffmpeg","libgomp1","libssl-dev","git","cmake","build-essential","python3-venv"],check=True)
BACKEND_SHA256="c2f3870a707eac9ea85f38e6cfb3e116e445ff9d8bb1efa1d4c5384cae551afe"
backend_bytes=base64.b64decode("...")
BACKEND_ARCHIVE=ROOT/"auk_backend_v030_transplant_candidate_2026-10-18_scopefix.zip";BACKEND_ARCHIVE.write_bytes(backend_bytes)
with zipfile.ZipFile(io.BytesIO(backend_bytes)) as z:
    for info in z.infolist():
        if not (ROOT/info.filename).resolve().is_relative_to(ROOT) or ((info.external_attr >> 16)&0o170000)==0o120000:raise RuntimeError("Unsafe backend ZIP")
    z.extractall(ROOT)
BACKEND_DIR=ROOT/"auk_reliability_backend"
import shutil; shutil.rmtree(BACKEND_DIR/"build", ignore_errors=True)  # never reuse stale build files
subprocess.run([sys.executable,"-m","pip","install","--no-deps",str(BACKEND_DIR)],check=True)
if "AUK_WORKER" in globals():AUK_WORKER.close();del AUK_WORKER
for name in list(sys.modules):
    if name=="auk_reliability" or name.startswith("auk_reliability."):del sys.modules[name]
import auk_reliability
assert auk_reliability.__version__=="0.3.0"
from auk_reliability import worker as _w, durable as _d
assert "whole_clip" in pathlib.Path(_w.__file__).read_text() and "only=None" in pathlib.Path(_d.__file__).read_text(), "Stale backend files were installed"
CHECKPOINT = None  # a checkpoint object from before this cell may point at the OLD backup; run P0 again (new backup folder)
print("Installed",auk_reliability.__version__,"exportable archive:",BACKEND_ARCHIVE)


In [ ]:
# @title P0. Mount Drive, checkpoint, or restore (no generation)
ENABLE_DRIVE_BACKUP = True # @param {type:"boolean"}
RESTORE_CHECKPOINT = True # @param {type:"boolean"}
# Tick RESTORE_CHECKPOINT only on a recycled runtime. It reads RESTORE_FROM and never writes to it. BACKUP_DIR must be a different, new folder.
RESTORE_FROM = "/content/drive/MyDrive/auk-checkpoints/session-2026-10-16-v025-b" # @param {type:"string"}
BACKUP_DIR = "/content/drive/MyDrive/auk-checkpoints/session-2026-10-17-repro" # @param {type:"string"}
DATASET_DIR = "/content/auk-public-v1" # @param {type:"string"}
RUNS_DIR = "/content/auk-runs" # @param {type:"string"}
CHECKPOINT = None
if ENABLE_DRIVE_BACKUP:
    from google.colab import drive
    from auk_reliability.durable import Checkpoint
    drive.mount('/content/drive')
    if RESTORE_CHECKPOINT:
        if __import__('os').path.realpath(RESTORE_FROM) == __import__('os').path.realpath(BACKUP_DIR): raise ValueError('BACKUP_DIR must be a different, new folder from RESTORE_FROM')
        # v0.2.5: restore only the corpus/runs from the old backup. The backend comes from P1, never from the old backup.
        old = Checkpoint(RESTORE_FROM, {"dataset":DATASET_DIR,"runs":RUNS_DIR})
        print(old.restore(only=("dataset","runs")))
        # v0.2.6: also bring back earlier request folders (hash-checked copy; skips identical files; stops if a local file differs).
        import json as _j, hashlib as _h
        _src = __import__("pathlib").Path(RESTORE_FROM)
        _recs = [r for r in _j.loads((_src/"checkpoint.json").read_text())["files"] if r["path"].startswith("backend/shared-requests/")]
        def _sha(p):
            h = _h.sha256()
            with open(p, "rb") as f:
                for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
            return h.hexdigest()
        _copied = 0
        for r in _recs:
            rel = r["path"].split("/", 1)[1]
            if ".." in rel.split("/"): raise ValueError("unsafe path")
            sp, tp = _src/r["path"], ROOT/rel
            if _sha(sp) != r["sha256"]: raise ValueError("Corrupt backup file " + r["path"])
            if tp.exists():
                if _sha(tp) != r["sha256"]: raise FileExistsError("Local file differs: " + str(tp))
                continue
            tp.parent.mkdir(parents=True, exist_ok=True); __import__("shutil").copyfile(sp, tp); _copied += 1
        print({"request_files_restored": _copied, "listed": len(_recs)})
        import glob, shutil, pathlib as _pl
        tools = sorted(glob.glob(RESTORE_FROM + "/backend/recovery-tools/*"))
        if tools:
            for f in _pl.Path(tools[-1]).glob("*.py"):
                if not (_pl.Path("/content")/f.name).exists(): shutil.copyfile(f, _pl.Path("/content")/f.name)
            print("Recovery tools copied from", tools[-1])
    CHECKPOINT = Checkpoint(BACKUP_DIR, {"backend":ROOT,"dataset":DATASET_DIR,"runs":RUNS_DIR})
    print(CHECKPOINT.sync())
else: print("Enable before generation. Pick a unique backup folder for a new experiment; reuse its exact path to restore. Virtualenvs/models are excluded and must be rebuilt/downloaded.")


In [ ]:
# @title P2. Fresh T4 CUDA compatibility, pinned runtime and three model profiles (downloads, no generation)
from auk_reliability.core import Limits, gpu_details, setup_runtime, setup_models, command, atomic_json
import site, os, subprocess, sys
LIMITS=Limits();GPU=gpu_details();print(GPU)
if GPU["compute_capability"]!="7.5":raise RuntimeError("Select a T4 runtime for this prebuilt. Other GPUs require explicit source-build validation.")
if GPU["total_mib"]<14000:raise RuntimeError("At least roughly 16 GiB total VRAM required")
# audio.cpp SM75 release uses CUDA 12.8, including on Colab images that ship CUDA 13.
subprocess.run([sys.executable,"-m","pip","install","--quiet","nvidia-cublas-cu12==12.8.4.1","nvidia-cuda-runtime-cu12==12.8.90","nvidia-cufft-cu12==11.3.3.83"],check=True)
search=set(site.getsitepackages()+[site.getusersitepackages()]);libdirs=[]
for root in search:
    for component in ("cublas","cuda_runtime","cufft"):
        p=pathlib.Path(root)/"nvidia"/component/"lib"
        if p.is_dir():libdirs.append(str(p))
os.environ["LD_LIBRARY_PATH"]=":".join(libdirs+[os.environ.get("LD_LIBRARY_PATH","")])
MEMORY_SAVER=True
BINARY,RUNTIME_LOCK=setup_runtime(ROOT,mode="t4_prebuilt",gpu=GPU)
link_check=command(["ldd",str(BINARY)],timeout=30)
if "not found" in link_check:raise RuntimeError("Runtime has unresolved libraries: "+link_check)
MODEL_DIR,MODEL_LOCK=setup_models(ROOT,["base_f32","base_q8","flash_f16"])
PROVENANCE={"runtime":RUNTIME_LOCK,"model":MODEL_LOCK,"environment":{"python":sys.version,"gpu":GPU,"nvidia_smi":command(["nvidia-smi"]),"ffmpeg":command(["ffmpeg","-version"]).splitlines()[0],"backend_sha256":BACKEND_SHA256}}
atomic_json(ROOT/"environment.json",PROVENANCE)
print("Ready. No model generation started. First request includes cold startup.")


{'name': 'Tesla T4', 'total_mib': 15360, 'free_mib': 14913, 'compute_capability': '7.5'}
Checking/downloading auk-base-f32.gguf (5.70 GiB)...
Checking/downloading auk-base-q8_0.gguf (1.53 GiB)...
Checking/downloading auk-flash-f16.gguf (2.85 GiB)...
Checking/downloading auk-vae-f32.gguf (0.59 GiB)...
Checking/downloading qwen2.5-omni-3b-q8_0.gguf (3.96 GiB)...
Ready. No model generation started. First request includes cold startup.


In [ ]:
# @title P3. Run CPU regression tests (no model generation)
subprocess.run([sys.executable,"-m","unittest","discover","-s",str(BACKEND_DIR/"tests"),"-v"],check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/auk-production/auk_reliability_backend/tests', '-v'], returncode=0)

In [ ]:
# @title P4. Configure shared worker, ASR and speaker QA (downloads, no generation)
ASR_LANGUAGE = "en"  # @param {type:"string"}
SETUP_SHARED_WORKER = True  # @param {type:"boolean"}
ENABLE_SPEAKER_QA = True  # @param {type:"boolean"}
GPU_HOURLY_RATE_USD = -1.0  # @param {type:"number"}
if SETUP_SHARED_WORKER:
    _need = [n for n in ("ROOT","LIMITS","GPU","MEMORY_SAVER","ASR_LANGUAGE","BINARY","MODEL_DIR") if n not in globals()]
    if _need:
        raise RuntimeError("Run P1 and P2 first (missing " + ", ".join(_need) + "). ")
    if "PROVENANCE" not in globals():
        PROVENANCE = {"runtime": RUNTIME_LOCK, "model": MODEL_LOCK, "environment": {"python":sys.version}}
    if "auk_reliability" not in sys.modules:
        raise RuntimeError("Run P1 first.")
    from auk_reliability.core import LocalAuKServer, setup_asr
    from auk_reliability.colab import setup_optional_speaker_qa
    from auk_reliability.qa import SemanticQAGate
    from auk_reliability.worker import ReliabilityWorker, configure_default_worker, edit_audio
    from auk_reliability.acceptance import configure_acceptance_suite, run_acceptance_suite, benchmark_profiles, recommend_profiles
    if "AUK_WORKER" in globals(): AUK_WORKER.close()
    shared_asr, shared_asr_fingerprint = setup_asr(ROOT / "shared-qa", LIMITS, ASR_LANGUAGE)
    speaker_callback, speaker_fingerprint, speaker_setup = setup_optional_speaker_qa(
        ROOT / "shared-speaker-qa", enabled=ENABLE_SPEAKER_QA)
    print("Speaker QA:", speaker_setup)
    if speaker_callback is None:
        raise RuntimeError("Speaker QA unavailable. Fix setup before reliability runs; unavailable identity QA never passes.")
    transport = LocalAuKServer(BINARY, MODEL_DIR, ROOT / "shared-runtime", LIMITS, MEMORY_SAVER)
    runtime_profile = {**PROVENANCE, "gpu": GPU, "simulation": False,
        "asr_fingerprint": shared_asr_fingerprint, "speaker_fingerprint": speaker_fingerprint, "speaker_setup": speaker_setup}
    if GPU_HOURLY_RATE_USD >= 0: runtime_profile["gpu_hourly_rate_usd"] = GPU_HOURLY_RATE_USD
    AUK_WORKER = ReliabilityWorker(transport, ROOT / "shared-requests",
        semantic_gate=SemanticQAGate(shared_asr, speaker_callback), runtime_profile=runtime_profile)
    configure_default_worker(AUK_WORKER)
    print("Shared worker configured. No generation started. Attribute QA still needs human review.")
else:
    print("Shared worker setup is off. Enable when ready for QA downloads/install.")


Speaker QA: {'status': 'ready', 'model_revision': '0f99f2d0ebe89ac095bcc5903c4dd8f72b367286'}
Shared worker configured. No generation started. Attribute QA still needs human review.


## Source review and progression
P5: use the rights-cleared `auk_public_corpus_v1.zip`, not the older two-clip example dataset. A restored corpus is reused only after checksum verification. P0 checkpoints source audio, corpus ZIP, CSVs, manifests, backend and request/run artifacts; models and virtualenvs are excluded. Backup is explicit, not background polling. A request interrupted before checkpoint may need to run again; completed verified rows resume without regeneration.

Review real development sources using `review_sources.py` from the corpus if available, or a player + CSV editor. Never mark transcripts, rights, labels or alignment checked without evidence. Replace/insert/delete need internal word boundaries with an untouched prefix/suffix. Whole-clip recipes do not test localized editing seams. Stay within current 15s input, 20s output, 30s combined budget; a 0.8x slowdown needs <=13.33s input. Only supported languages belong in the manifest.

P6 dev valid -> P7 eight different checked recordings, one per operation -> P9 listen and record actual pass/fail -> P8 balanced dev 30/op/profile -> P9 all selected outputs -> P10 strict candidates -> P6 held-out + speaker/source disjoint checks -> P11 100+ sources/1,000+ requests -> P9 -> P12 -> fresh-runtime reproduction. If quality fails, diagnose/regression-test and rerun a NEW dev experiment. Never tune on held-out results and still call the same set untouched.

Attribute/naturalness QA remains human-reviewed. Automatic ASR and speaker similarity alone cannot establish emotion, confidence, naturalness, speed or splice quality. Fix recurring failures or swap the model; do not promise perfect unseen outputs.


In [ ]:
import json
# @title P5. Extract corpus once or verify/reuse restored corpus (no inference)
UPLOAD_CORPUS = True # @param {type:"boolean"}
REUSE_EXISTING_CORPUS = True # @param {type:"boolean"}
CORPUS_ZIP = "/content/auk_public_corpus_v1.zip" # @param {type:"string"}
DATASET_DIR = "/content/auk-public-v1" # @param {type:"string"}
from auk_reliability.workflow import extract_bundle
from auk_reliability.core import sha256_file, atomic_json
if UPLOAD_CORPUS:
    archive=pathlib.Path(CORPUS_ZIP)
    if REUSE_EXISTING_CORPUS and not archive.is_file():archive=pathlib.Path(DATASET_DIR)/"source_corpus.zip"
    if not archive.is_file():
        from google.colab import files
        files.upload()
    if not archive.is_file():raise FileNotFoundError(CORPUS_ZIP)
    marker=pathlib.Path(DATASET_DIR)/"corpus_integrity.json"
    if REUSE_EXISTING_CORPUS:
        saved=json.loads(marker.read_text())
        if saved["archive_sha256"]!=sha256_file(archive):raise RuntimeError("Wrong corpus ZIP")
        for rel,digest in saved["source_hashes"].items():
            if sha256_file(pathlib.Path(DATASET_DIR)/rel)!=digest:raise RuntimeError("Changed source: "+rel)
        print("Verified restored corpus. Reviewed CSVs are preserved.")
    else:
        data_root=extract_bundle(archive,DATASET_DIR)
        shutil_copy=__import__('shutil').copy2
        shutil_copy(archive,data_root/"source_corpus.zip")
        from auk_reliability.workflow import AUDIO_EXTENSIONS
        atomic_json(marker,{"archive_sha256":sha256_file(archive),"source_hashes":{str(p.relative_to(data_root)):sha256_file(p) for p in data_root.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS}})
        print("Extracted. Review development sources/CSV before P6; never batch-tick review flags.")
    if CHECKPOINT:print(CHECKPOINT.sync())
else:print("Set UPLOAD_CORPUS only when the correct rights-cleared corpus ZIP is ready.")


In [ ]:
# @title P6. Build distinct dev/held-out manifests and show the correct preflight (no inference)
BUILD_FROM_CSV = True # @param {type:"boolean"}
# whole_clip (v0.2.5): transcript edits regenerate the whole <=15 s utterance, like the take-home notebook; no hand-checked
# boundaries needed (exact ASR match of the whole clip is still required). localized: original reviewed-boundary mode (tests splice seams).
TRANSCRIPT_SCOPE = "whole_clip" # @param ["whole_clip", "localized"]
BUILD_SPLIT = "development" # @param ["development", "held_out"]
DEVELOPMENT_CSV = "/content/auk-public-v1/clips_development.csv" # @param {type:"string"}
HELD_OUT_CSV = "/content/auk-public-v1/clips_held_out.csv" # @param {type:"string"}
DEV_MANIFEST = "/content/auk-public-v1/development_manifest.json" # @param {type:"string"}
HELD_OUT_MANIFEST = "/content/auk-public-v1/held_out_manifest.json" # @param {type:"string"}
COVERAGE_REVIEWER = "Advay" # @param {type:"string"}
SUPPORTED_LANGUAGES = "en" # @param {type:"string"}
from auk_reliability.dataset import validate_dataset
from auk_reliability.workflow import build_manifest
from auk_reliability.durable import assert_split_disjoint
ACTIVE_WORKFLOWS=("voice_editor","transcript_editor")
if BUILD_FROM_CSV:
    csv_path=DEVELOPMENT_CSV if BUILD_SPLIT=="development" else HELD_OUT_CSV
    if TRANSCRIPT_SCOPE=="whole_clip":
        from auk_reliability.workflow import make_whole_clip_csv
        _wc=pathlib.Path(csv_path).with_name(pathlib.Path(csv_path).stem+"_wholeclip.csv")
        if not _wc.exists(): print(make_whole_clip_csv(csv_path,_wc))
        csv_path=str(_wc)
        print("Building from", csv_path, "(any rows listed above under rows_left_unchanged keep reviewed localized boundaries)")
    manifest=DEV_MANIFEST if BUILD_SPLIT=="development" else HELD_OUT_MANIFEST
    build_manifest(csv_path,manifest,reviewer=COVERAGE_REVIEWER,languages=[x.strip() for x in SUPPORTED_LANGUAGES.split(',') if x.strip()])
    audit=validate_dataset(manifest,acceptance=BUILD_SPLIT=="held_out",modes=ACTIVE_WORKFLOWS)
    expected={"dev","development"} if BUILD_SPLIT=="development" else {"held_out"}
    if any(c['split'] not in expected for c in audit['clips'].values()):raise ValueError("Wrong split in CSV")
    print(BUILD_SPLIT,"valid:",audit["valid"],"sources:",audit["distinct_sources"],"requests:",audit["request_count"])
    print("Errors:",audit["errors"][:40]);print("Coverage:",audit["levels"])
    if pathlib.Path(DEV_MANIFEST).is_file() and pathlib.Path(HELD_OUT_MANIFEST).is_file():assert_split_disjoint(DEV_MANIFEST,HELD_OUT_MANIFEST)
    if CHECKPOINT:print(CHECKPOINT.sync())
else:print("Build development first; held-out stays untouched until settings are frozen.")


In [ ]:
%run /content/diagnose_segment.py

In [ ]:
%run /content/diagnose_full_source.py

In [ ]:
import tempfile, pathlib
from IPython.display import Audio, display
from auk_reliability.core import prepare_audio, compare_transcript
with tempfile.TemporaryDirectory() as d:
    crop = pathlib.Path(d)/'candidate.wav'
    prepare_audio(SOURCE_FILE, crop, 1.23, 2.05, AUK_WORKER.limits)
    display(Audio(data=crop.read_bytes()))
    observed = AUK_WORKER.gate.transcriber(crop)
    text = observed.get('text','') if isinstance(observed,dict) else observed
    print('Candidate ASR:', observed)
    print('Comparison:', compare_transcript('the better team',text))

In [ ]:
print(CHECKPOINT.sync())

In [ ]:
import pathlib, shutil, datetime
recovery = CHECKPOINT.roots['backend']/'recovery-tools'/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
recovery.mkdir(parents=True, exist_ok=False)
for name in ('review_sources_v2.py', 'fix_checkpoint_speaker_cache.py', 'diagnose_segment.py', 'diagnose_full_source.py'):
    source = pathlib.Path('/content')/name
    if source.is_file():
        shutil.copy2(source, recovery/name)
        print('Preserved:', name)
    else:
        print('Not present:', name)
print(CHECKPOINT.sync())
print('Recovery tools saved at:', recovery)

In [ ]:
# @title P7. One real smoke request (explicitly opt in: tick RUN_SHARED_EDIT and SHARED_RIGHTS_CONFIRMED yourself)
INPUT_AUDIO = "/content/auk-public-v1/audio/vctk-p326_028-mic1.flac"  # @param {type:"string"}
RUN_SHARED_EDIT = True  # @param {type:"boolean"}
SHARED_MODE = "transcript_editor"  # @param ["voice_editor", "transcript_editor"]
# voice_editor: emotion | voice_characteristics | whisper | speed | custom
# transcript_editor: replace | insert | delete
SHARED_OPERATION = "delete"  # @param ["emotion", "voice_characteristics", "whisper", "speed", "custom", "replace", "insert", "delete"]
SHARED_PROFILE = "base_f32"  # @param ["base_f32", "base_q8", "flash_f16"]
SHARED_RIGHTS_CONFIRMED = True  # @param {type:"boolean"}
SHARED_SEED = 42  # @param {type:"integer"}
# whole_clip (v0.2.5): edit the whole <=15 s recording, prompt "Replace 'old' with 'new'." like the take-home. Needs no segment times.
# localized: your original checked segment times below.
TRANSCRIPT_SCOPE = "whole_clip"  # @param ["whole_clip", "localized"]
CHECKED_SOURCE_TEXT = "It is believed he had been stabbed."  # @param {type:"string"}
# Per-operation settings. Edit only the ones for your chosen SHARED_OPERATION.
EMOTION = "calm"  # @param {type:"string"}
VOICE_DESCRIPTION = "warm, slightly deeper voice"  # @param {type:"string"}
SPEED_FACTOR = 1.25  # @param {type:"number"}
CUSTOM_INSTRUCTION = "Keep every word but sound more confident."  # @param {type:"string"}
# Transcript edits need a listened-to, nonempty interval (max 15 s) and exact checked text inside it.
SEGMENT_START_SECONDS = 1.23  # @param {type:"number"}
SEGMENT_END_SECONDS = 2.05  # @param {type:"number"}
SEGMENT_TEXT = "the better team"  # @param {type:"string"}
# replace: phrase to replace / delete: phrase to remove / insert: text to add. Type it WITHOUT quote marks.
EDIT_TARGET_TEXT = "It is"  # @param {type:"string"}
# replace only. Type it WITHOUT quote marks.
EDIT_REPLACEMENT_TEXT = ""  # @param {type:"string"}
INSERT_ANCHOR = "believed"  # @param {type:"string"}
INSERT_POSITION = "before"  # @param ["before", "after"]

def _clean(v): return "" if v is None else str(v).strip().strip("\"'\u201c\u201d").strip()

def build_shared_parameters():
    global EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT
    EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT = _clean(EDIT_TARGET_TEXT), _clean(EDIT_REPLACEMENT_TEXT)
    base = {"source_text": CHECKED_SOURCE_TEXT, "rights_confirmed": SHARED_RIGHTS_CONFIRMED,
            "profile": SHARED_PROFILE, "seed": SHARED_SEED}
    op = SHARED_OPERATION
    if SHARED_MODE == "voice_editor":
        if op not in ("emotion", "voice_characteristics", "whisper", "speed", "custom"):
            raise RuntimeError(f"{op} is a transcript operation. Set SHARED_MODE = 'transcript_editor'.")
        if op == "emotion": base["emotion"] = EMOTION
        elif op == "voice_characteristics": base["description"] = VOICE_DESCRIPTION
        elif op == "speed": base["speed_factor"] = SPEED_FACTOR
        elif op == "custom":
            base["instruction"] = CUSTOM_INSTRUCTION
            base["semantic_contract"] = {"preserve_transcript": True}
        return base
    if op not in ("replace", "insert", "delete"):
        raise RuntimeError(f"{op} is a voice-editor operation. Set SHARED_MODE = 'voice_editor'.")
    if not SEGMENT_TEXT.strip() or not EDIT_TARGET_TEXT.strip():
        raise RuntimeError("Transcript edits need SEGMENT_TEXT and EDIT_TARGET_TEXT (checked by listening).")
    if TRANSCRIPT_SCOPE == "whole_clip":
        seg = CHECKED_SOURCE_TEXT
        base.update(segment_start_seconds=0, segment_end_seconds=None, segment_text=seg, segment_scope="whole_clip")
    else:
        seg = SEGMENT_TEXT
        base.update(segment_start_seconds=SEGMENT_START_SECONDS, segment_end_seconds=SEGMENT_END_SECONDS,
                    segment_text=seg, alignment_verified=True)
    if op == "replace":
        if not EDIT_REPLACEMENT_TEXT.strip(): raise RuntimeError("Set EDIT_REPLACEMENT_TEXT.")
        base.update(original_text=EDIT_TARGET_TEXT, replacement_text=EDIT_REPLACEMENT_TEXT,
                    edited_segment_text=seg.replace(EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT, 1))
    elif op == "delete":
        base.update(text=EDIT_TARGET_TEXT,
                    edited_segment_text=" ".join(seg.replace(EDIT_TARGET_TEXT, "", 1).split()))
    else:
        if not INSERT_ANCHOR.strip(): raise RuntimeError("Set INSERT_ANCHOR (a phrase that occurs once in SEGMENT_TEXT).")
        base.update(text=EDIT_TARGET_TEXT, anchor=INSERT_ANCHOR, position=INSERT_POSITION,
                    edited_segment_text=(EDIT_TARGET_TEXT + " " + INSERT_ANCHOR) if INSERT_POSITION == "before" else (INSERT_ANCHOR + " " + EDIT_TARGET_TEXT))
        base["edited_segment_text"] = seg.replace(INSERT_ANCHOR, base["edited_segment_text"], 1)
    return base

if RUN_SHARED_EDIT:
    if CHECKPOINT is None:raise RuntimeError("Enable and verify P0 Drive checkpoint before generation.")
    if SHARED_MODE not in ("voice_editor", "transcript_editor"):
        raise RuntimeError("Voice Enhancer is out of scope and owned by another engineer.")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run P4 first.")
    SOURCE_FILE = pathlib.Path(INPUT_AUDIO) if INPUT_AUDIO.strip() else None
    if SOURCE_FILE is None or not SOURCE_FILE.is_file(): raise RuntimeError("Set INPUT_AUDIO to an uploaded source file.")
    SHARED_PARAMETERS = build_shared_parameters()
    print("ABOUT TO RUN (check before it uses the GPU):", SHARED_OPERATION, "| scope:", TRANSCRIPT_SCOPE if SHARED_MODE == "transcript_editor" else "n/a")
    print("  source file:", SOURCE_FILE.name)
    for _k in ("source_text", "original_text", "replacement_text", "text", "anchor", "position", "edited_segment_text"):
        if _k in SHARED_PARAMETERS: print(f"  {_k}: {SHARED_PARAMETERS[_k]!r}")
    try:
        shared_result = edit_audio(SOURCE_FILE, SHARED_MODE, SHARED_OPERATION, SHARED_PARAMETERS)
    finally:
        print(CHECKPOINT.sync())
    print("RESULT:", shared_result.outcome, "| reason:", shared_result.reason, "| technical_success:", shared_result.technical_success,
          "| automatic_qa_pass:", shared_result.automatic_qa_pass, "| latency_s:", round(shared_result.latency_seconds))
    print("request_id:", shared_result.request_id, "(full details are synced to the Drive checkpoint)")
    for _a in shared_result.attempts:
        _t = (_a.get("qa") or {}).get("checks", {}).get("transcript", {})
        _sp = (_a.get("qa") or {}).get("checks", {}).get("speaker", {})
        print(f"  attempt {_a['attempt']} seed {_a['seed']}: ASR error {_t.get('error_rate')} | speaker {_sp.get('score')} | heard:", repr(_t.get("observed")))
    if shared_result.output_path:
        from IPython.display import Audio, display
        display(Audio(filename=shared_result.output_path))
    print("Human requested-effect/naturalness/seam review remains required.")


In [ ]:
print(CHECKPOINT.sync())

In [ ]:
# @title P8. Speaker-balanced profile benchmark with verified resume (real GPU workload, off)
RUN_PROFILE_BENCHMARK = True # @param {type:"boolean"}
RESUME_BENCHMARK = False # @param {type:"boolean"}
BENCHMARK_DIR = "/content/auk-runs/benchmark-2026-10-16-01" # @param {type:"string"}
PER_OPERATION_LIMIT = 1 # @param {type:"integer"}
if RUN_PROFILE_BENCHMARK:
    if CHECKPOINT is None:raise RuntimeError("Verify P0 checkpoint first")
    if "AUK_WORKER" not in globals():raise RuntimeError("Run P4 first")
    from auk_reliability.acceptance import benchmark_profiles
    benchmark=benchmark_profiles(AUK_WORKER,DEV_MANIFEST,BENCHMARK_DIR,per_operation_limit=PER_OPERATION_LIMIT,modes=ACTIVE_WORKFLOWS,resume=RESUME_BENCHMARK,checkpoint=CHECKPOINT)
    print("Coverage:",benchmark["coverage"])
    print("Review every selected result in P9. At least 30/op/profile for P10; no shortcut for bad audio.")
else:print("First test eight distinct-source smoke cases in P7. 30 x 8 x 3 = up to 720 requests before retries, not an instruction to start now.")


In [ ]:
# @title P7d. Targeted discriminator round for the transcript-family failures (real GPU workload, off)
RUN_DISCRIMINATOR = False # @param {type:"boolean"}
DISCRIMINATOR_DIR = "/content/auk-runs/discriminator-2026-10-17-01" # @param {type:"string"}
WALL_CLOCK_BUDGET_MINUTES = 75 # @param {type:"integer"}
RESUME_DISCRIMINATOR = True # @param {type:"boolean"}
if RUN_DISCRIMINATOR:
    if CHECKPOINT is None: raise RuntimeError("Verify P0 checkpoint first")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run P4 first")
    import auk_reliability, dataclasses, time, json as _json, os as _os
    assert auk_reliability.__version__=="0.3.0", "Run P1 first: the discriminator needs the v0.3.0 time guard and explicit-insert arm"
    from pathlib import Path as _P
    from auk_reliability.core import atomic_json, WallClockAbort
    from auk_reliability.durable import add_artifact_hashes
    from auk_reliability.acceptance import load_review, environment
    _dm=(_json.load(open(DEV_MANIFEST)) if isinstance(DEV_MANIFEST,str) else DEV_MANIFEST)
    _clips=(_dm.get("clips",_dm) if isinstance(_dm,dict) else _dm)
    clip=(_clips.get("vctk-p225_025-mic1") if isinstance(_clips,dict) else next(c for c in _clips if c["id"]=="vctk-p225_025-mic1"))
    CLIP=clip["audio_path"]
    COMMON={"source_text":"Why do you want to come to Edinburgh?","rights_confirmed":True,
            "segment_start_seconds":0.0,"segment_end_seconds":2.92025,
            "segment_text":"Why do you want to come to Edinburgh?","alignment_verified":False,
            "segment_scope":"whole_clip"}
    DELETE={**COMMON,"text":"Why do","edited_segment_text":"you want to come to Edinburgh?"}
    INSERT={**COMMON,"text":"really","anchor":"Why do","position":"after","edited_segment_text":"Why do really you want to come to Edinburgh?"}
    REPLACE={**COMMON,"original_text":"Why do","replacement_text":"Hello there","edited_segment_text":"Hello there you want to come to Edinburgh?"}
    REQUESTS=[]
    for seed in (1000,2000):
        for op,prof,params in (("delete","base_f32",DELETE),("delete","base_q8",DELETE),
                               ("insert","base_f32",INSERT),("insert","base_q8",INSERT),
                               ("insert","flash_f16",INSERT),("replace","flash_f16",REPLACE)):
            REQUESTS.append(("A","transcript_editor",op,prof,seed,params))
        for prof in ("base_f32","base_q8","flash_f16"):
            REQUESTS.append(("B","transcript_editor","insert",prof,seed,{**INSERT,"instruction_style":"explicit_insert"}))
        for op,prof,params in (("delete","flash_f16",DELETE),("replace","base_f32",REPLACE),("replace","base_q8",REPLACE)):
            REQUESTS.append(("C","transcript_editor",op,prof,seed,params))
    out=_P(DISCRIMINATOR_DIR);out.mkdir(parents=True,exist_ok=True)
    progress=out/"discriminator_progress.jsonl"
    rows=[_json.loads(l) for l in progress.read_text().splitlines()] if (RESUME_DISCRIMINATOR and progress.exists()) else []
    done={(r["arm"],r["operation"],r["profile"],r["seed_base"]) for r in rows}
    AUK_WORKER.set_deadline(WALL_CLOCK_BUDGET_MINUTES*60)
    stopped_early=None
    try:
        for arm,mode,op,prof,seed,params in REQUESTS:
            if (arm,op,prof,seed) in done: continue
            began=time.monotonic()
            try:
                result=AUK_WORKER.edit_audio(CLIP,mode,op,{**params,"profile":prof,"seed":seed})
                row=dataclasses.asdict(result);row["semantic_success"]=load_review(result.request_dir,result)
            except WallClockAbort:
                stopped_early="wall_clock_budget";break
            except Exception as e:
                row={"technical_success":False,"automatic_qa_pass":False,"semantic_success":None,
                     "latency_seconds":time.monotonic()-began,"reason":getattr(e,"code",type(e).__name__),
                     "request_dir":getattr(e,"request_dir",None),"attempts":[],"output_path":None,"profile":prof,
                     "error_detail":str(e)[:1800]}
            row.update({"arm":arm,"mode":mode,"operation":op,"profile":prof,"seed_base":seed,
                        "instruction_style":params.get("instruction_style","replace_prompt"),
                        "case_id":f"disc-{op}-{prof}-s{seed}","clip_id":clip["id"]})
            add_artifact_hashes(row);rows.append(row)
            with progress.open("a") as fh: fh.write(_json.dumps(row,allow_nan=False)+"\n");fh.flush();_os.fsync(fh.fileno())
            atomic_json(out/"discriminator.json",{"rows":rows,"complete":False,"stopped_early":stopped_early,"environment":environment(AUK_WORKER)})
            print(f"arm {arm} {op}/{prof} seed {seed}: tech={row.get('technical_success')} qa={row.get('automatic_qa_pass')} reason={row.get('reason')}")
            CHECKPOINT.sync()
    finally:
        AUK_WORKER.set_deadline(None)
    if stopped_early: CHECKPOINT.sync()
    arms={}
    for r in rows:
        a=arms.setdefault(r["arm"],{"requests":0,"technical_success":0,"automatic_qa_pass":0})
        a["requests"]+=1;a["technical_success"]+=bool(r.get("technical_success"));a["automatic_qa_pass"]+=bool(r.get("automatic_qa_pass"))
    summary={"rows":rows,"complete":len(rows)==len(REQUESTS),"stopped_early":stopped_early,
             "arms":arms,"environment":environment(AUK_WORKER),
             "note":"Arm A: fresh-seed control re-run of the six P8 auto-QA failures. Arm B: explicit-insert phrasing (insert only). Arm C: fresh-seed regeneration of the three human-failed outputs (still needs human listening; semantic_success stays None until reviewed). Descriptive counts only - two seeds per cell cannot prove causation."}
    atomic_json(out/"discriminator.json",summary)
    CHECKPOINT.sync()
    print("Arms:",_json.dumps(arms,indent=1));print("Complete:",summary["complete"],"Stopped early:",stopped_early)
else: print("Off. Tick RUN_DISCRIMINATOR to run the 24-request targeted round (P8 measured ~44 GPU-min for 24 requests on this clip; the wall-clock guard stops it at the budget). Requires P1 v0.3.0, then P0 and P4.")


In [ ]:
# @title P7e. Voice-enhancer smoke through the same shared harness (real GPU workload, off)
RUN_ENHANCER_SMOKE = False # @param {type:"boolean"}
ENHANCER_DIR = "/content/auk-runs/enhancer-2026-10-17-01" # @param {type:"string"}
WALL_CLOCK_BUDGET_MINUTES = 40 # @param {type:"integer"}
RESUME_ENHANCER = True # @param {type:"boolean"}
if RUN_ENHANCER_SMOKE:
    if CHECKPOINT is None: raise RuntimeError("Verify P0 checkpoint first")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run P4 first")
    import auk_reliability, dataclasses, time, json as _json, os as _os
    assert auk_reliability.__version__=="0.3.0", "Run P1 first (v0.3.0 time guard)"
    from pathlib import Path as _P
    from auk_reliability.core import atomic_json, WallClockAbort
    from auk_reliability.durable import add_artifact_hashes
    from auk_reliability.acceptance import load_review, environment
    _dm=(_json.load(open(DEV_MANIFEST)) if isinstance(DEV_MANIFEST,str) else DEV_MANIFEST)
    _clips=(_dm.get("clips",_dm) if isinstance(_dm,dict) else _dm)
    clip=(_clips.get("vctk-p225_025-mic1") if isinstance(_clips,dict) else next(c for c in _clips if c["id"]=="vctk-p225_025-mic1"))
    CLIP=clip["audio_path"]
    # Same verified clip, transcript, rights and QA gates as the other two modes. voice_enhancer
    # operations are part of the shipped backend policy (separate product owner for the enhancer
    # deliverable; this cell only measures the shared-harness path, it does not change ownership).
    BASE={"source_text":"Why do you want to come to Edinburgh?","rights_confirmed":True}
    REQUESTS=[]
    for seed in (1000,2000):
        for op in ("enhance","denoise","dereverb"):
            for prof in ("base_f32","base_q8","flash_f16"):
                REQUESTS.append(("voice_enhancer",op,prof,seed,BASE))
    out=_P(ENHANCER_DIR);out.mkdir(parents=True,exist_ok=True)
    progress=out/"enhancer_progress.jsonl"
    rows=[_json.loads(l) for l in progress.read_text().splitlines()] if (RESUME_ENHANCER and progress.exists()) else []
    done={(r["operation"],r["profile"],r["seed_base"]) for r in rows}
    AUK_WORKER.set_deadline(WALL_CLOCK_BUDGET_MINUTES*60)
    stopped_early=None
    try:
        for mode,op,prof,seed,params in REQUESTS:
            if (op,prof,seed) in done: continue
            began=time.monotonic()
            try:
                result=AUK_WORKER.edit_audio(CLIP,mode,op,{**params,"profile":prof,"seed":seed})
                row=dataclasses.asdict(result);row["semantic_success"]=load_review(result.request_dir,result)
            except WallClockAbort:
                stopped_early="wall_clock_budget";break
            except Exception as e:
                row={"technical_success":False,"automatic_qa_pass":False,"semantic_success":None,
                     "latency_seconds":time.monotonic()-began,"reason":getattr(e,"code",type(e).__name__),
                     "request_dir":getattr(e,"request_dir",None),"attempts":[],"output_path":None,"profile":prof,
                     "error_detail":str(e)[:1800]}
            row.update({"mode":mode,"operation":op,"profile":prof,"seed_base":seed,
                        "case_id":f"enh-{op}-{prof}-s{seed}","clip_id":clip["id"]})
            add_artifact_hashes(row);rows.append(row)
            with progress.open("a") as fh: fh.write(_json.dumps(row,allow_nan=False)+"\n");fh.flush();_os.fsync(fh.fileno())
            atomic_json(out/"enhancer.json",{"rows":rows,"complete":False,"stopped_early":stopped_early,"environment":environment(AUK_WORKER)})
            print(f"{op}/{prof} seed {seed}: tech={row.get('technical_success')} qa={row.get('automatic_qa_pass')} reason={row.get('reason')}")
            CHECKPOINT.sync()
    finally:
        AUK_WORKER.set_deadline(None)
    if stopped_early: CHECKPOINT.sync()
    n=len(rows); qa=sum(bool(r.get("automatic_qa_pass")) for r in rows)
    summary={"rows":rows,"complete":n==len(REQUESTS),"stopped_early":stopped_early,
             "requests":n,"automatic_qa_pass":qa,"environment":environment(AUK_WORKER),
             "note":"Voice-enhancer smoke through the same worker, QA gates and recording as the other modes. Human listening still required before any semantic claim; semantic_success stays None until reviewed."}
    atomic_json(out/"enhancer.json",summary)
    CHECKPOINT.sync()
    print(f"Enhancer smoke: {qa}/{n} automatic-QA pass. Complete: {summary['complete']}. Stopped early: {stopped_early}")
else: print("Off. Tick RUN_ENHANCER_SMOKE for 18 requests (3 enhancer ops x 3 profiles x 2 seeds, ~35 GPU-min). Requires P1 v0.3.0, P0, P4.")


In [ ]:
# @title P7f. Insert anchor-placement follow-up (real GPU workload, off)
RUN_INSERT_FOLLOWUP = True # @param {type:"boolean"}
FOLLOWUP_DIR = "/content/auk-runs/discriminator-fix-2026-10-17-01" # @param {type:"string"}
WALL_CLOCK_BUDGET_MINUTES = 35 # @param {type:"integer"}
RESUME_FOLLOWUP = True # @param {type:"boolean"}
if RUN_INSERT_FOLLOWUP:
    if CHECKPOINT is None: raise RuntimeError("Verify P0 checkpoint first")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run P4 first")
    import auk_reliability, dataclasses, time, json as _json, os as _os
    assert auk_reliability.__version__=="0.3.0", "Run P1 first"
    from pathlib import Path as _P
    from auk_reliability.core import atomic_json, WallClockAbort
    from auk_reliability.durable import add_artifact_hashes
    from auk_reliability.acceptance import load_review, environment
    _dm=(_json.load(open(DEV_MANIFEST)) if isinstance(DEV_MANIFEST,str) else DEV_MANIFEST)
    _clips=(_dm.get("clips",_dm) if isinstance(_dm,dict) else _dm)
    clip=(_clips.get("vctk-p225_025-mic1") if isinstance(_clips,dict) else next(c for c in _clips if c["id"]=="vctk-p225_025-mic1"))
    CLIP=clip["audio_path"]
    COMMON={"source_text":"Why do you want to come to Edinburgh?","rights_confirmed":True,
            "segment_start_seconds":0.0,"segment_end_seconds":2.92025,
            "segment_text":"Why do you want to come to Edinburgh?","alignment_verified":False,
            "segment_scope":"whole_clip"}
    # Natural placement: insert 'really' after 'you' -> "Why do you really want ..." (the placement the model itself produced once in Arm B)
    NATURAL={**COMMON,"text":"really","anchor":"you","position":"after","edited_segment_text":"Why do you really want to come to Edinburgh?"}
    REQUESTS=[]
    for seed in (1000,2000):
        for prof in ("base_f32","flash_f16"):
            REQUESTS.append(("D","transcript_editor","insert",prof,seed,NATURAL))
        REQUESTS.append(("E","transcript_editor","insert","base_f32",seed,{**NATURAL,"instruction_style":"explicit_insert"}))
    out=_P(FOLLOWUP_DIR);out.mkdir(parents=True,exist_ok=True)
    progress=out/"followup_progress.jsonl"
    rows=[_json.loads(l) for l in progress.read_text().splitlines()] if (RESUME_FOLLOWUP and progress.exists()) else []
    done={(r["arm"],r["operation"],r["profile"],r["seed_base"]) for r in rows}
    AUK_WORKER.set_deadline(WALL_CLOCK_BUDGET_MINUTES*60)
    stopped_early=None
    try:
        for arm,mode,op,prof,seed,params in REQUESTS:
            if (arm,op,prof,seed) in done: continue
            began=time.monotonic()
            try:
                result=AUK_WORKER.edit_audio(CLIP,mode,op,{**params,"profile":prof,"seed":seed})
                row=dataclasses.asdict(result);row["semantic_success"]=load_review(result.request_dir,result)
            except WallClockAbort:
                stopped_early="wall_clock_budget";break
            except Exception as e:
                row={"technical_success":False,"automatic_qa_pass":False,"semantic_success":None,
                     "latency_seconds":time.monotonic()-began,"reason":getattr(e,"code",type(e).__name__),
                     "request_dir":getattr(e,"request_dir",None),"attempts":[],"output_path":None,"profile":prof,
                     "error_detail":str(e)[:1800]}
            row.update({"arm":arm,"mode":mode,"operation":op,"profile":prof,"seed_base":seed,
                        "instruction_style":params.get("instruction_style","replace_prompt"),
                        "case_id":f"fix-{op}-{prof}-s{seed}","clip_id":clip["id"]})
            add_artifact_hashes(row);rows.append(row)
            with progress.open("a") as fh: fh.write(_json.dumps(row,allow_nan=False)+"\n");fh.flush();_os.fsync(fh.fileno())
            CHECKPOINT.sync()
            print(f"arm {arm} {op}/{prof} seed {seed}: tech={row['technical_success']} qa={row['automatic_qa_pass']} reason={row.get('reason')}",flush=True)
    finally:
        AUK_WORKER.set_deadline(None)
    if stopped_early: CHECKPOINT.sync()
    arms={}
    for r in rows:
        a=arms.setdefault(r["arm"],{"requests":0,"technical_success":0,"automatic_qa_pass":0})
        a["requests"]+=1;a["technical_success"]+=bool(r.get("technical_success"));a["automatic_qa_pass"]+=bool(r.get("automatic_qa_pass"))
    summary={"rows":rows,"complete":len(rows)==len(REQUESTS),"stopped_early":stopped_early,
             "arms":arms,"environment":environment(AUK_WORKER),
             "note":"Arm D/E: insert 'really' after 'you' (natural placement) vs the original awkward anchor. Descriptive, 2 seeds."}
    atomic_json(out/"followup.json",summary)
    CHECKPOINT.sync()
    print("Arms:",_json.dumps(arms,indent=1));print("Complete:",len(rows)==len(REQUESTS),"Stopped early:",stopped_early)
else: print("Off. Tick RUN_INSERT_FOLLOWUP to run the 12-request follow-up round.")


In [ ]:
# @title P7h. Reduced held-out acceptance smoke (24 requests, 3 per op, real GPU workload, off)
RUN_HELDOUT_SMOKE = False # @param {type:"boolean"}
HELDOUT_BUNDLE_DIR = "/content/auk-heldout/auk-heldout" # @param {type:"string"}
HELDOUT_DIR = "/content/auk-runs/heldout-2026-10-17-01" # @param {type:"string"}
WALL_CLOCK_BUDGET_MINUTES = 150 # @param {type:"integer"}
RESUME_HELDOUT = True # @param {type:"boolean"}
REQUESTS_PER_OP = 3 # @param {type:"integer"}
if RUN_HELDOUT_SMOKE:
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run P4 first")
    import auk_reliability, dataclasses, time, json as _json, os as _os, hashlib as _hl, glob as _g
    assert auk_reliability.__version__=="0.3.0", "Run P1 first (v0.3.0 time guard)"
    from pathlib import Path as _P
    from auk_reliability.core import atomic_json, WallClockAbort
    from auk_reliability.durable import add_artifact_hashes
    from auk_reliability.acceptance import load_review, environment
    # NOTE: no P0 checkpoint/Drive dependency by design. This smoke reads the sha-verified
    # held-out bundle transferred onto the runtime and writes locally; results are copied off
    # through the existing authorized upload route afterwards.
    _mani=_json.load(open(_P(HELDOUT_BUNDLE_DIR)/"held_out_acceptance_manifest.json"))
    _clips={c["id"]:c for c in _mani["clips"]}
    # Verify corpus integrity on this runtime before any request: sha256 every clip named by
    # the manifest against the manifest's own recorded hash.
    _bad=[]
    for _cid,_c in _clips.items():
        _hits=_g.glob(str(_P(HELDOUT_BUNDLE_DIR)/"audio"/(_cid+".*")))
        if not _hits: _bad.append((_cid,"missing")); continue
        if _hl.sha256(open(_hits[0],"rb").read()).hexdigest()!=_c["sha256"]: _bad.append((_cid,"hash"))
    if _bad: raise RuntimeError(f"Corpus integrity failed for {len(_bad)} clips: {_bad[:5]}")
    print(f"Corpus integrity OK: {len(_clips)} manifest clips hash-verified on runtime")
    # Documented reduction: first REQUESTS_PER_OP requests per operation, in manifest order.
    _sel={}
    for _r in _mani["requests"]:
        _sel.setdefault(_r["operation"],[])
        if len(_sel[_r["operation"]])<REQUESTS_PER_OP: _sel[_r["operation"]].append(_r)
    REQUESTS=[_r for _op in _sel for _r in _sel[_op]]
    out=_P(HELDOUT_DIR);out.mkdir(parents=True,exist_ok=True)
    progress=out/"heldout_progress.jsonl"
    rows=[_json.loads(l) for l in progress.read_text().splitlines()] if (RESUME_HELDOUT and progress.exists()) else []
    done={r["request_id"] for r in rows}
    AUK_WORKER.set_deadline(WALL_CLOCK_BUDGET_MINUTES*60)
    stopped_early=None
    try:
        for _r in REQUESTS:
            if _r["id"] in done: continue
            _clip=_clips[_r["clip_id"]]
            CLIP=_g.glob(str(_P(HELDOUT_BUNDLE_DIR)/"audio"/(_r["clip_id"]+".*")))[0]
            began=time.monotonic()
            try:
                result=AUK_WORKER.edit_audio(CLIP,_r["mode"],_r["operation"],dict(_r["parameters"]))
                row=dataclasses.asdict(result);row["semantic_success"]=load_review(result.request_dir,result)
            except WallClockAbort:
                stopped_early="wall_clock_budget";break
            except Exception as e:
                row={"technical_success":False,"automatic_qa_pass":False,"semantic_success":None,
                     "latency_seconds":time.monotonic()-began,"reason":getattr(e,"code",type(e).__name__),
                     "request_dir":getattr(e,"request_dir",None),"attempts":[],"output_path":None,
                     "error_detail":str(e)[:1800]}
            row.update({"mode":_r["mode"],"operation":_r["operation"],"request_id":_r["id"],
                        "clip_id":_r["clip_id"],"seed_base":_r["parameters"].get("seed"),
                        "profile":_r["parameters"].get("profile")})
            add_artifact_hashes(row);rows.append(row)
            with progress.open("a") as fh: fh.write(_json.dumps(row,allow_nan=False)+"\n");fh.flush();_os.fsync(fh.fileno())
            atomic_json(out/"heldout.json",{"rows":rows,"complete":False,"stopped_early":stopped_early,"environment":environment(AUK_WORKER)})
            print(f"{_r['operation']} {_r['id']}: tech={row.get('technical_success')} qa={row.get('automatic_qa_pass')} reason={row.get('reason')}")
    finally:
        AUK_WORKER.set_deadline(None)
    n=len(rows); qa=sum(bool(r.get("automatic_qa_pass")) for r in rows)
    from collections import Counter as _C
    per_op={op:{"n":0,"qa":0} for op in _sel}
    for r in rows:
        per_op[r["operation"]]["n"]+=1; per_op[r["operation"]]["qa"]+=int(bool(r.get("automatic_qa_pass")))
    summary={"rows":rows,"complete":n==len(REQUESTS),"stopped_early":stopped_early,
             "requests":n,"automatic_qa_pass":qa,"per_operation":per_op,"environment":environment(AUK_WORKER),
             "note":"REDUCED held-out acceptance smoke: first 3 requests per operation in manifest order (24 of 1000), corpus sha-verified on runtime, no config re-qualification. Human listening still required before any semantic claim; semantic_success stays None until reviewed. This smoke does NOT complete the full held-out benchmark and does not close the known unresolved items (insert no-op, naturalness detection gap, localized generated-quality gap, narrow-benchmark limits)."}
    atomic_json(out/"heldout.json",summary)
    print(f"Held-out smoke: {qa}/{n} automatic-QA pass. Complete: {summary['complete']}. Stopped early: {stopped_early}")
else: print("Off. Tick RUN_HELDOUT_SMOKE for the reduced held-out smoke (24 requests = first 3 per op in manifest order, ~1 GPU-hour). Requires P1 v0.3.0, P2 models, P4 worker. No P0/Drive needed: reads the sha-verified bundle at HELDOUT_BUNDLE_DIR and writes locally.")


In [ ]:
# @title P9. Listen and save one hash-bound quality review (no inference)
REQUEST_DIR = "/content/auk-production/shared-requests/b3f53981dbc04d2f8a2916c5fedeebe9"  # @param {type:"string"}
REVIEWER = "Advay"  # @param {type:"string"}
SAVE_REVIEW = True  # @param {type:"boolean"}
QUALITY_PASSED = True  # @param {type:"boolean"}
REVIEW_NOTES = ""  # @param {type:"string"}
if REQUEST_DIR.strip():
    import json
    from IPython.display import Audio, display
    folder=pathlib.Path(REQUEST_DIR);result=json.loads((folder/"result.json").read_text())
    print("Technical:",result["technical_success"],"auto QA:",result["automatic_qa_pass"],"reason:",result["reason"])
    if (folder/"input.wav").is_file():display(Audio(filename=str(folder/"input.wav")))
    if result["output_path"]:display(Audio(filename=result["output_path"]))
    print("Check words, target effect, identity where preserved, speed/timing, naturalness, seams and endings. Review every selected benchmark/acceptance output.")
    if SAVE_REVIEW:
        from auk_reliability.workflow import save_review
        print(save_review(folder,reviewer=REVIEWER,passed=QUALITY_PASSED,notes=REVIEW_NOTES))
else:print("Use request_dir from benchmark.json or acceptance requests.csv.")

if SAVE_REVIEW and CHECKPOINT:print(CHECKPOINT.sync())


In [ ]:
# @title P9b. Review queue - unreviewed outputs and QA-failure inspection (no inference)
MODE = "review" # @param ["review", "inspect_failures"]
QUEUE_ROOT = "/content/auk-production/shared-requests" # @param {type:"string"}
REVIEWER = "Advay" # @param {type:"string"}
VERDICTS = "ec6768a4:fail, 60970ddf:fail, 68da0e36:fail, f94ad3c0:pass, 0ccb9de9:pass, c25a2141:pass, f82a72ae:pass, 0cacac57:pass, e054bf66:pass, b4c4345b:pass, 32e0322a:pass, 9a80a05b:pass, d64c3eb3:pass" # @param {type:"string"} - review mode: comma list like b3f53981:pass, 64befcdd:fail
FAILURE_NOTES = "" # @param {type:"string"} - inspect_failures mode: comma list like abc123:sounds fine despite QA, def456:robotic
import json as _json, dataclasses, pathlib, hashlib
from IPython.display import Audio, display
from auk_reliability.workflow import save_review
from auk_reliability.acceptance import load_review
from auk_reliability.worker import Result
CHECKPOINT = globals().get("CHECKPOINT")
def _sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def _last_artifact(row):
    for path in reversed((row.get("attempts") or [{}])[-1].get("artifact_paths",[])):
        if path.endswith(".wav") and pathlib.Path(path).is_file():return path
    return None
rows=[]
for folder in sorted(pathlib.Path(QUEUE_ROOT).iterdir()):
    rf=folder/"result.json"
    if not rf.is_file():continue
    row=_json.loads(rf.read_text());row["_folder"]=folder
    rows.append(row)
if MODE=="review":
    queue=[]
    for row in rows:
        folder=row["_folder"]
        allowed={f.name for f in dataclasses.fields(Result)}
        result=Result(**{k:v for k,v in row.items() if k in allowed})
        if not row.get("automatic_qa_pass"):continue
        if load_review(folder,result) is not None:continue
        queue.append((folder,row))
    print(len(queue),"output(s) awaiting human review.")
    for folder,row in queue:
        plan=(row.get("attempts") or [{}])[-1].get("plan",{})
        print("-"*72)
        print(folder.name,"|",row.get("profile"),"|",plan.get("public_operation") or plan.get("operation"),"|",plan.get("instruction"))
        print("  expected:",plan.get("expected_transcript"))
        if (folder/"input.wav").is_file():display(Audio(filename=str(folder/"input.wav")))
        if row.get("output_path") and pathlib.Path(row["output_path"]).is_file():display(Audio(filename=row["output_path"]))
    saved=[]
    for token in [t.strip() for t in VERDICTS.split(",") if t.strip()]:
        rid,_,verdict=token.partition(":")
        match=[f for f,_ in queue if f.name.startswith(rid)]
        if len(match)!=1 or verdict not in ("pass","fail"):raise ValueError("Bad verdict token: "+repr(token))
        saved.append(save_review(match[0],reviewer=REVIEWER,passed=(verdict=="pass")))
    for s in saved:print("saved:",s)
    if saved and CHECKPOINT:print(CHECKPOINT.sync())
else:
    pending=[]
    for row in rows:
        folder=row["_folder"]
        if row.get("automatic_qa_pass"):continue
        if (folder/"failure_observation.json").is_file():continue
        pending.append((folder,row))
    print(len(pending),"QA-failed request(s) without a failure note.")
    print("These remain automatic-QA failures no matter how they sound. Notes inform debugging; they never flip a result.")
    for folder,row in pending:
        plan=(row.get("attempts") or [{}])[-1].get("plan",{})
        qa=(row.get("attempts") or [{}])[-1].get("qa",{})
        print("-"*72)
        print(folder.name,"|",row.get("profile"),"|",plan.get("public_operation") or plan.get("operation"),"| QA:",row.get("reason"))
        print("  expected:",plan.get("expected_transcript"),"| qa detail:",_json.dumps(qa.get("checks",{}))[:220])
        artifact=_last_artifact(row)
        if artifact:display(Audio(filename=artifact))
        else:print("  (no playable artifact)")
    saved=[]
    for token in [t.strip() for t in FAILURE_NOTES.split(",") if t.strip()]:
        rid,_,note=token.partition(":")
        note=note.strip()
        match=[(f,r) for f,r in pending if f.name.startswith(rid)]
        if len(match)!=1 or not note:raise ValueError("Bad failure-note token: "+repr(token))
        folder,row=match[0];artifact=_last_artifact(row)
        rec={"reviewer":REVIEWER,"note":note,"qa_failure_reason":row.get("reason"),
             "artifact_sha256":_sha(artifact) if artifact else None,
             "recorded_at":__import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
             "effect":"observation only; the automatic QA failure stands"}
        (folder/"failure_observation.json").write_text(_json.dumps(rec,indent=2));saved.append(rec)
    for s in saved:print("saved note:",s)
    if saved and CHECKPOINT:print(CHECKPOINT.sync())

In [ ]:
# @title Export checkpoint for delivery review (no inference)
import pathlib, json, zipfile, hashlib
from google.colab import files
checkpoint_root = pathlib.Path('/content/drive/MyDrive/auk-checkpoints/session-2026-10-16-v025-b')
checkpoint_manifest = json.loads((checkpoint_root/'checkpoint.json').read_text())
export_path = pathlib.Path('/content/auk_checkpoint_for_review.zip')
with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(checkpoint_root/'checkpoint.json', 'checkpoint.json')
    for record in checkpoint_manifest['files']:
        relative = pathlib.Path(record['path'])
        source = (checkpoint_root/relative).resolve()
        if relative.is_absolute() or not source.is_relative_to(checkpoint_root.resolve()):
            raise ValueError('Unsafe checkpoint path')
        data = source.read_bytes()
        if hashlib.sha256(data).hexdigest() != record['sha256']:
            raise ValueError('Checkpoint integrity mismatch: '+str(relative))
        archive.writestr(str(relative), data)
print('Verified files:', len(checkpoint_manifest['files']))
print('Archive bytes:', export_path.stat().st_size)
print('Archive SHA256:', hashlib.sha256(export_path.read_bytes()).hexdigest())
files.download(str(export_path))


In [ ]:
# @title P6b. Verify transcripts by listening (human ground truth, no inference)
VERIFY_CSV = "/content/auk-public-v1/clips_held_out.csv" # @param {type:"string"}
VERDICTS_DIR = "/content/auk-public-v1/transcript-verdicts" # @param {type:"string"}
REVIEWER = "Advay" # @param {type:"string"}
BATCH = 25 # @param {type:"integer"}
VERDICT_LINES = "vctk-p231_032-mic2: ok; vctk-p313_027-mic1: ok; vctk-p313_028-mic2: ok; vctk-p313_029-mic1: ok; vctk-p313_030-mic2: ok; vctk-p252_025-mic2: ok; vctk-p252_027-mic2: ok; vctk-p252_028-mic1: ok; vctk-p252_029-mic2: ok; vctk-p363_026-mic2: ok; vctk-p363_029-mic1: ok; vctk-p363_031-mic1: ok; vctk-p363_032-mic2: ok; vctk-p232_032-mic1: ok; vctk-p340_026-mic1: ok" # @param {type:"string"} - one per line or separated by ; : clip_id: ok   OR   clip_id: corrected words here
REPAIR_CLIP = "vctk-p299_025-mic1" # @param {type:"string"} - repairs a verdict corrupted by flattened batch input; skips when already clean
WRITE_VERIFIED_CSV = True # @param {type:"boolean"}
VERIFIED_CSV_OUT = "/content/auk-public-v1/clips_held_out_verified.csv" # @param {type:"string"}
import csv, json, hashlib, datetime, pathlib, re
from IPython.display import Audio, display
def _sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def _load_verdicts(vdir):
    out={}
    for f in pathlib.Path(vdir).glob('*.json'):
        try:v=json.loads(f.read_text())
        except Exception:continue
        if v.get('clip_id') and v.get('audio_sha256') and v.get('final_transcript') and v.get('reviewer'):
            out[v['clip_id']]=v
    return out
csv_path=pathlib.Path(VERIFY_CSV);vdir=pathlib.Path(VERDICTS_DIR);vdir.mkdir(parents=True,exist_ok=True)
if not csv_path.is_file():
    print(f"No transcript CSV at {csv_path} - stage the held-out dataset first (cell P5). Nothing to verify.")
    raise SystemExit
rows=list(csv.DictReader(csv_path.open(newline='',encoding='utf-8-sig')))
verdicts=_load_verdicts(vdir)
pending=[]
for r in rows:
    audio=csv_path.parent/r['audio_path']
    v=verdicts.get(r['id'])
    if v and audio.is_file() and v['audio_sha256']==_sha(audio):continue
    pending.append((r,audio))
print(f"{len(rows)-len(pending)} of {len(rows)} clips have a valid verdict. {len(pending)} remaining.")
_ID_TOKEN=re.compile(r'[\w-]+_\d+-mic\d\s*:')
_repaired=False
def _write_verified_csv():
    verified=0;fieldnames=list(rows[0].keys())
    if 'transcript_verified' not in fieldnames:fieldnames.append('transcript_verified')
    out=[]
    for r in rows:
        audio=csv_path.parent/r['audio_path'];v=verdicts.get(r['id'])
        r=dict(r)
        if v and audio.is_file() and v['audio_sha256']==_sha(audio):
            r['transcript']=v['final_transcript'];r['transcript_verified']='true';verified+=1
        else:r['transcript_verified']='false'
        out.append(r)
    with open(VERIFIED_CSV_OUT,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames);w.writeheader();w.writerows(out)
    print(f"Wrote {VERIFIED_CSV_OUT}: {verified} verified, {len(out)-verified} still unverified.")
if REPAIR_CLIP.strip():
    _target=REPAIR_CLIP.strip()
    _old=verdicts.get(_target)
    if _old is None:raise ValueError(f"No saved verdict to repair for {_target!r}")
    if _ID_TOKEN.search(_old.get('final_transcript','')):
        _m=[r for r in rows if r['id']==_target]
        if len(_m)!=1:raise ValueError(f"Unknown clip id in REPAIR_CLIP: {_target!r}")
        _audio=csv_path.parent/_m[0]['audio_path']
        if not _audio.is_file() or _old['audio_sha256']!=_sha(_audio):raise ValueError(f"Audio changed since the original verdict for {_target!r}; repair blocked")
        if not REVIEWER.strip():raise ValueError('Name the reviewer')
        _rec={'clip_id':_target,'audio_sha256':_old['audio_sha256'],'draft_transcript':_m[0]['transcript'].strip(),
             'final_transcript':_m[0]['transcript'].strip(),'verdict':'confirmed','reviewer':REVIEWER,
             'reviewed_at':_old.get('reviewed_at'),
             'repaired_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
             'repair_reason':'batch input flattened multiple verdicts into one line; owner confirmed the draft transcript matched',
             'repaired_from':_old}
        (vdir/(_target+'.json')).write_text(json.dumps(_rec,indent=2));verdicts[_target]=_rec
        _repaired=True
        print(f"Repaired {_target}: verdict recorded as confirmed against the draft transcript; the corrupt record is kept under 'repaired_from'.")
    else:
        print(f"REPAIR_CLIP {_target}: saved verdict is already clean; skipping repair (human verdicts are never overridden).")
parsed=[]
for line in [l.strip() for l in re.split(r'[\n;]+', VERDICT_LINES) if l.strip()]:
    cid,_,text=line.partition(':')
    cid=cid.strip();text=text.strip()
    match=[(r,a) for r,a in pending if r['id']==cid]
    if len(match)!=1:raise ValueError(f"Unknown or already-verified clip id in verdict line: {cid!r}")
    r,audio=match[0]
    if not audio.is_file():raise ValueError(f"Missing audio for {cid}")
    draft=r['transcript'].strip()
    if text.lower()=='ok':final,verdict=draft,'confirmed'
    elif len(text.split())>=2:
        if _ID_TOKEN.search(text):raise ValueError(f"Verdict for {cid} looks like several verdicts flattened into one line - separate verdicts with semicolons, one per clip")
        final,verdict=text,'corrected'
    else:raise ValueError(f"Verdict for {cid} must be 'ok' or the corrected full transcript")
    parsed.append((cid,audio,draft,final,verdict))
if not REVIEWER.strip():raise ValueError('Name the reviewer')
saved=0
for cid,audio,draft,final,verdict in parsed:
    rec={'clip_id':cid,'audio_sha256':_sha(audio),'draft_transcript':draft,'final_transcript':final,
         'verdict':verdict,'reviewer':REVIEWER,'reviewed_at':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    (vdir/(cid+'.json')).write_text(json.dumps(rec,indent=2));verdicts[cid]=rec;saved+=1
if saved:
    print(f"Saved {saved} verdict(s).")
    pending=[(r,csv_path.parent/r['audio_path']) for r in rows
             if not (r['id'] in verdicts and (csv_path.parent/r['audio_path']).is_file()
                     and verdicts[r['id']]['audio_sha256']==_sha(csv_path.parent/r['audio_path']))]
    print(f"{len(pending)} remaining.")
if WRITE_VERIFIED_CSV or (_repaired and pathlib.Path(VERIFIED_CSV_OUT).is_file()):
    _write_verified_csv()
for r,audio in pending[:BATCH]:
    print('-'*72)
    print(f"{r['id']}  ({r['accent']}, {r['microphone']})")
    print('draft:',r['transcript'])
    if audio.is_file():display(Audio(filename=str(audio)))
    else:print('AUDIO MISSING - do not verdict this clip')
print("After listening: put verdicts in VERDICT_LINES, one per line or separated by semicolons, e.g.")
print("  vctk-p299_025-mic1: ok; vctk-p299_026-mic2: I just went for it over the weekend")
print("Then rerun this cell. Tick WRITE_VERIFIED_CSV only when all remaining counts look right.")
CHECKPOINT = globals().get("CHECKPOINT")
if CHECKPOINT:print(CHECKPOINT.sync())

In [ ]:
# @title P10. Reviewed screening candidates + speaker-disjoint held-out manifest (no inference)
SELECT_CANDIDATES = False # @param {type:"boolean"}
SELECTED_MANIFEST = "/content/auk-public-v1/selected_manifest.json" # @param {type:"string"}
if SELECT_CANDIDATES:
    from auk_reliability.workflow import refresh_benchmark,candidate_profiles,apply_candidates
    from auk_reliability.durable import assert_split_disjoint
    assert_split_disjoint(DEV_MANIFEST,HELD_OUT_MANIFEST)
    audit=validate_dataset(HELD_OUT_MANIFEST,modes=ACTIVE_WORKFLOWS)
    if not audit['valid']:raise ValueError(audit['errors'][:40])
    reviewed_benchmark=refresh_benchmark(pathlib.Path(BENCHMARK_DIR)/'benchmark.json')
    CANDIDATES=candidate_profiles(reviewed_benchmark,require_perfect=True)
    print({k:{"profile":v["recommended_profile"],"status":v["status"]} for k,v in CANDIDATES.items()})
    apply_candidates(HELD_OUT_MANIFEST,CANDIDATES,SELECTED_MANIFEST)
    if CHECKPOINT:print(CHECKPOINT.sync())
else:print("100%-observed reviewed dev gate. A missing or failed candidate blocks selection; do not lower gates.")


In [ ]:
# @title P11. Held-out acceptance with verified resume (real GPU workload, off)
RUN_ACCEPTANCE = False # @param {type:"boolean"}
RESUME_ACCEPTANCE = False # @param {type:"boolean"}
ACCEPTANCE_RUN_DIR = "/content/auk-runs/acceptance-2026-10-13-01" # @param {type:"string"}
if RUN_ACCEPTANCE:
    if CHECKPOINT is None:raise RuntimeError("Verify P0 checkpoint first")
    if "AUK_WORKER" not in globals():raise RuntimeError("Run P4 first")
    from auk_reliability.acceptance import run_acceptance_suite
    from auk_reliability.durable import assert_split_disjoint
    assert_split_disjoint(DEV_MANIFEST,HELD_OUT_MANIFEST)
    summary=run_acceptance_suite(AUK_WORKER,SELECTED_MANIFEST,ACCEPTANCE_RUN_DIR,modes=ACTIVE_WORKFLOWS,resume=RESUME_ACCEPTANCE,checkpoint=CHECKPOINT)
    print(summary['overall']);print("Technical gate:",summary['technical_acceptance_pass'],"Observed perfect gate:",summary['observed_perfect_pass'])
    print("Listen/review every selected output in P9; refresh reports in P12. Unknown review never passes.")
else:print("Freeze settings before held-out testing. Use a new run folder after any configuration/manifest change.")


In [ ]:
# @title P12. Refresh reviews, full dated export and measured handoff (no inference)
EXPORT_DELIVERABLES = False # @param {type:"boolean"}
if EXPORT_DELIVERABLES:
    import shutil,json,datetime
    from auk_reliability.acceptance import refresh_reviews
    from auk_reliability.workflow import write_handoff
    summary=refresh_reviews(ACCEPTANCE_RUN_DIR)
    config=write_handoff(ACCEPTANCE_RUN_DIR,pathlib.Path(BENCHMARK_DIR)/'benchmark.json',pathlib.Path(ACCEPTANCE_RUN_DIR)/'handoff')
    print("Technical gate:",summary['technical_acceptance_pass'],"Observed perfect gate:",summary['observed_perfect_pass'])
    export=ROOT/('deliverables-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'));export.mkdir()
    shutil.copy2(BACKEND_ARCHIVE,export/BACKEND_ARCHIVE.name)
    shutil.copytree(DATASET_DIR,export/'dataset')
    for directory,name in ((ACCEPTANCE_RUN_DIR,'acceptance'),(BENCHMARK_DIR,'benchmark'),(ROOT/'shared-requests','request-artifacts')):shutil.copytree(directory,export/name)
    shutil.copy2(ROOT/'environment.json',export/'setup_environment.json')
    config_example={"run_generation":False,"commercial_use":False,"commercial_license_confirmed":False,"work_dir":"/content/auk-production","dataset_dir":DATASET_DIR,"runs_dir":RUNS_DIR,"backup_dir":BACKUP_DIR,"manifest":SELECTED_MANIFEST,"output_dir":"/content/auk-runs/fresh-reproduction-01","language":"en","gpu_hourly_rate_usd":None}
    (export/'runner_config.example.json').write_text(json.dumps(config_example,indent=2))
    bundle=shutil.make_archive(str(export),'zip',export)
    if CHECKPOINT:print(CHECKPOINT.sync())
    from google.colab import files
    files.download(bundle)
    print("Export includes private source speech/transcripts/reviews. Export may contain failures; not a completion certificate. Save the updated notebook separately, then reproduce on a fresh T4 before handoff.")
else:print("Exports are dated and never merged into an old deliverable directory.")


# Legacy runner (original org content preserved)
The original cells below are unchanged for history and other engineers. Use P1-P12 above for the shared Voice Editor + Transcript Editor deliverables. Do not run this legacy path in parallel: it can start a second worker or show old instructions that do not apply to the new workflow.


# Magic Hour | AI Voice Editor
## Production-oriented Colab runner · AuK Base / Flash

Upload a voice recording, choose an edit, compare the result, and download a WAV. This replaces the fixed examples in `tencent.ipynb` with a reusable single-worker implementation.

**Start:** Select a **T4 GPU**, run setup through section 4, upload one file in section 5, then set the controls and run section 6. Do not use **Run all**: it would also re-run setup and model downloads. Run the numbered cells one at a time. No generation, benchmark, approval, export, or deletion runs until its checkbox is enabled.

**Included:** immutable model revision; SHA-256 checks; versioned runtime; real Base/Flash profiles; short-clip processing; one job at a time; SQLite status and idempotency; bounded subprocess and HTTP requests; hung-process termination; input/output decoding checks; optional CPU transcription checks; explicit human review; cold/warm benchmarks; private results and diagnostics.

**Scope:** voice editing, not general music/SFX editing. At most **15 seconds of input** per job, **20 seconds of output**, and a conservative **30-second combined budget**. Longer files require an explicit range. The result is the selected clip only, in mono 24 kHz; this notebook does not silently trim a file, splice a segment into a long recording, or remux/lip-sync a video.

**Validation:** 64 local tests passed on CPU, including real local-HTTP fixtures and simulated failures. No actual AuK or Whisper model inference was executed during notebook construction. Run the GPU acceptance checklist at the end before rollout. Quantized/Flash output quality is not established by a successful WAV export.

**Deployment boundary:** this is a hardened, single-user runner and production handoff candidate, not a multi-tenant service or a guarantee that Colab is suitable hosting. Job recovery lasts only while its local disk survives. Production still needs durable object storage/queueing, authentication and asset ownership, usage/billing integration, isolation, retention policy, and GPU acceptance tests.

**License:** Qwen2.5-Omni-3B remains part of this stack. Its published license permits research/evaluation and requires a separate license for commercial use. The commercial switch below stays blocked without explicit clearance. A checkbox does not obtain a license. See the source notes at the end.

## 1. Settings
`base_f32` preserves your notebook's generator/VAE precision; the Qwen conditioner remains Q8. `flash_f16` is a separate distilled checkpoint, not Base with fewer steps. `base_q8` is an explicit compact candidate, never a hidden fallback.

Keep `t4_prebuilt` on T4. The official release build targets SM75. Other GPUs require the explicit `source_build` path and a CUDA development toolkit. `auto` uses memory-saving mode when available VRAM is below 22 GiB; this is a conservative heuristic, not a capacity guarantee. Resident mode must be benchmarked on the actual GPU.

In [ ]:
# @title Configuration
from pathlib import Path
import os
import sys

WORK_DIR = "/content/auk-voice-editor"  # @param {type:"string"}
RUNTIME_MODE = "t4_prebuilt"  # @param ["t4_prebuilt", "source_build"]
PROFILE = "base_f32"  # @param ["base_f32", "flash_f16", "base_q8"]
DOWNLOAD_BASE_AND_FLASH = True  # @param {type:"boolean"}
MEMORY_MODE = "auto"  # @param ["auto", "save_memory", "resident"]
ENABLE_ASR_QA = False  # @param {type:"boolean"}
ASR_LANGUAGE = "auto"  # @param {type:"string"}
COMMERCIAL_USE = False  # @param {type:"boolean"}
COMMERCIAL_LICENSE_CONFIRMED = False  # @param {type:"boolean"}

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ Colab runtime.")
if COMMERCIAL_USE and not COMMERCIAL_LICENSE_CONFIRMED:
    raise RuntimeError("Commercial use requires independently confirmed model-license clearance.")
if "EDITOR" in globals() and EDITOR is not None:
    EDITOR.close()
    EDITOR = None
ROOT = Path(WORK_DIR).expanduser().resolve()
if str(ROOT) in ("/", "/content", "/tmp", str(Path.home())):
    raise ValueError("Use a dedicated subdirectory, not a filesystem or home root.")
ROOT.mkdir(parents=True, exist_ok=True)
os.chmod(ROOT, 0o700)
os.umask(0o077)
print("Working directory:", ROOT)
print("No generation is started by this cell.")


### Define the runner
These cells define functions only. They do not download weights, start a GPU process, or change your audio. They are separated for inspection and backend reuse.

In [ ]:
# @title Core types, configuration, and edit planning
from __future__ import annotations

import array
import base64
import contextlib
import dataclasses
import datetime as dt
import fcntl
import hashlib
import json
import math
import os
import re
import shutil
import signal
import socket
import sqlite3
import subprocess
import sys
import tarfile
import tempfile
import threading
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import uuid
from pathlib import Path
from typing import Any, Callable

CORE_VERSION = '1.0.0'
MODEL_REPO = 'audio-cpp/AuK-Base-and-Flash-GGUF'
MODEL_REVISION = '2814a16a49b822a5709e11a78cfd7c1662f19b61'
RUNTIME_TAG = 'v0.9.0'
RUNTIME_COMMIT_PREFIX = '795c45f'
RUNTIME_ASSET = 'audio-v0.9.0-bin-ubuntu-x64-cuda12.8-colab.tar.gz'
RUNTIME_SHA256 = 'c9ed906f918246669c324f0d31f1b7dd80cbe003c35cf8a54932f333b57ca3f6'
RUNTIME_URL = f'https://github.com/0xShug0/audio.cpp/releases/download/{RUNTIME_TAG}/{RUNTIME_ASSET}'
PROFILE_FILES = {
    'base_f32': ('base', 'auk-base-f32.gguf'),
    'flash_f16': ('flash', 'auk-flash-f16.gguf'),
    'base_q8': ('base', 'auk-base-q8_0.gguf'),
}
SHARED_FILES = ('qwen2.5-omni-3b-q8_0.gguf', 'auk-vae-f32.gguf')
SIDECARS = ('config/auk-base.yaml', 'config/auk-flash.yaml',
            'tokenizer/tokenizer.json', 'tokenizer/tokenizer_config.json')
ALLOWED_SUFFIXES = {'.wav', '.mp3', '.flac', '.m4a', '.mp4', '.mov', '.ogg', '.opus', '.webm'}
FORMAT_WHITELIST = 'wav,mp3,flac,mov,matroska,webm,ogg,aac'

class EditorError(RuntimeError):
    def __init__(self, code: str, message: str):
        self.code = code
        super().__init__(message)


def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def canonical_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    allow_nan=False, separators=(',', ':')).encode()).hexdigest()


def atomic_bytes(path: Path, content: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix='.' + path.name + '.', dir=path.parent)
    try:
        with os.fdopen(fd, 'wb') as stream:
            stream.write(content)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
        os.chmod(path, 0o600)
    finally:
        Path(temporary).unlink(missing_ok=True)


def atomic_json(path: Path, content: Any) -> None:
    atomic_bytes(path, json.dumps(content, ensure_ascii=False, indent=2,
                                 allow_nan=False).encode('utf-8'))


def command(args: list[str], *, timeout: float = 60, cwd: Path | None = None,
            binary: bool = False) -> Any:
    """Never invokes a shell; user-supplied filenames cannot become commands."""
    try:
        result = subprocess.run([str(a) for a in args], cwd=cwd, timeout=timeout,
                                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, check=False)
    except subprocess.TimeoutExpired as exc:
        raise EditorError('process_timeout', f'{Path(args[0]).name} exceeded {timeout:g}s.') from exc
    except OSError as exc:
        raise EditorError('process_unavailable', f'Could not start {Path(args[0]).name}: {exc}') from exc
    if result.returncode:
        # Bounded diagnostic, no successful subprocess output is placed in job manifests.
        detail = result.stderr.decode(errors='replace')[-1800:]
        raise EditorError('process_failed', f'{Path(args[0]).name} exited {result.returncode}: {detail}')
    return result.stdout if binary else result.stdout.decode('utf-8', errors='replace')


def require_finite(value: float, name: str, low: float, high: float) -> float:
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise EditorError('invalid_argument', f'{name} must be a number.') from exc
    if not math.isfinite(number) or not low <= number <= high:
        raise EditorError('invalid_argument', f'{name} must be between {low} and {high}.')
    return number


@dataclasses.dataclass(frozen=True)
class Limits:
    max_upload_bytes: int = 64 * 1024 * 1024
    max_source_seconds: float = 600.0
    max_input_seconds: float = 15.0
    max_output_seconds: float = 20.0
    combined_seconds: float = 30.0  # Conservative application policy, not an audio.cpp guarantee.
    min_audio_seconds: float = 0.5
    ffmpeg_timeout: float = 45.0
    startup_timeout: float = 300.0
    inference_timeout: float = 900.0
    response_bytes: int = 8 * 1024 * 1024
    asr_timeout: float = 180.0


@dataclasses.dataclass(frozen=True)
class EditSpec:
    operation: str = 'custom'
    instruction: str = ''
    emotion: str = 'excited'
    voice_description: str = ''
    old_words: str = ''
    new_words: str = ''
    speed: float = 1.0
    target_seconds: float | None = None
    source_transcript: str = ''
    expected_transcript: str = ''
    seed: int = 42


def build_plan(spec: EditSpec, duration: float, limits: Limits) -> dict:
    duration = require_finite(duration, 'input duration', limits.min_audio_seconds, limits.max_input_seconds)
    if isinstance(spec.seed, bool) or not isinstance(spec.seed, int) or not 0 <= spec.seed < 2**31:
        raise EditorError('invalid_seed', 'Use an integer seed from 0 to 2147483647.')
    for name in ('instruction', 'voice_description', 'old_words', 'new_words',
                 'source_transcript', 'expected_transcript', 'emotion'):
        value = getattr(spec, name)
        if not isinstance(value, str) or len(value) > 4000 or '\x00' in value:
            raise EditorError('invalid_text', f'{name} must be text of at most 4,000 characters without NUL.')
    op = spec.operation
    preserve_content = op in {'emotion', 'whisper', 'enhance', 'denoise', 'dereverb', 'voice', 'speed'}
    prompts = {
        'whisper': 'Speak in a soft whisper, retaining the words and speaker identity.',
        'enhance': 'Clean the speech by reducing noise and room reverberation. Keep all speakers, words, and duration.',
        'denoise': 'Reduce background noise while retaining all speech and the original duration.',
        'dereverb': 'Reduce room reverberation without changing the speech, speaker, or duration.',
    }
    expected = spec.expected_transcript.strip()
    target = duration
    if op == 'emotion':
        if spec.emotion not in {'happy', 'angry', 'sad', 'fearful', 'surprised', 'disgusted', 'calm', 'excited'}:
            raise EditorError('invalid_emotion', 'Choose one of the emotions listed in the form.')
        prompt = f'Change the emotion to {spec.emotion}. Retain the words and speaker identity.'
    elif op in prompts:
        prompt = prompts[op]
    elif op == 'voice':
        if not spec.voice_description.strip():
            raise EditorError('missing_description', 'Enter a voice description, not a named person.')
        prompt = f'Keep the words unchanged. Change the voice character to {json.dumps(spec.voice_description)}.'
    elif op == 'speed':
        speed = require_finite(spec.speed, 'speed', 0.5, 2.0)
        target = duration / speed
        prompt = f'Adjust the speech speed to {speed:g}x.'
        if spec.target_seconds is not None and abs(spec.target_seconds - target) > .02:
            raise EditorError('duration_conflict', 'Speed determines duration. Leave target_seconds at 0 in the form.')
    elif op == 'replace_words':
        if not spec.old_words.strip() or not spec.new_words.strip() or spec.old_words == spec.new_words:
            raise EditorError('invalid_replacement', 'Provide different nonempty old and new words.')
        if not spec.source_transcript.strip():
            raise EditorError('missing_transcript', 'Word replacement requires a human-checked source transcript.')
        if spec.source_transcript.count(spec.old_words) != 1:
            raise EditorError('ambiguous_replacement', 'The old words must occur exactly once in the source transcript. Select a smaller clip for repeated words.')
        derived = spec.source_transcript.replace(spec.old_words, spec.new_words, 1)
        if expected and expected != derived:
            raise EditorError('transcript_conflict', 'Expected transcript disagrees with the specified replacement.')
        expected = derived
        prompt = f'Replace {json.dumps(spec.old_words, ensure_ascii=False)} with {json.dumps(spec.new_words, ensure_ascii=False)}.'
    elif op == 'custom':
        prompt = spec.instruction.strip()
        if not prompt:
            raise EditorError('missing_instruction', 'Enter an editing instruction.')
    else:
        raise EditorError('unknown_operation', f'Unsupported operation: {op}')
    if spec.target_seconds is not None and op != 'speed':
        if op not in {'custom', 'replace_words'} and abs(spec.target_seconds - duration) > .02:
            raise EditorError('duration_conflict', 'This preset preserves duration. Use custom for a duration-changing edit.')
        target = require_finite(spec.target_seconds, 'target_seconds', limits.min_audio_seconds, limits.max_output_seconds)
    if preserve_content and spec.source_transcript.strip():
        if expected and expected != spec.source_transcript.strip():
            raise EditorError('transcript_conflict', 'This operation should preserve the source transcript.')
        expected = spec.source_transcript.strip()
    target = math.ceil((target - 1e-9) / .02) * .02
    require_finite(target, 'output duration', limits.min_audio_seconds, limits.max_output_seconds)
    if duration + target > limits.combined_seconds + .001:
        raise EditorError('duration_budget', f'Input + output exceeds the {limits.combined_seconds:g}s application budget. Select a shorter clip.')
    return {'operation': op, 'instruction': prompt, 'target_seconds': round(target, 3),
            'preserve_content': preserve_content, 'expected_transcript': expected, 'seed': spec.seed}


In [ ]:
# @title Bounded audio decoding and validation
# --- Bounded media decoding and measurable, non-semantic checks ---

def probe_audio(path: Path, limits: Limits, *, user_upload: bool = True) -> dict:
    path = Path(path).resolve()
    if not path.is_file() or path.stat().st_size == 0:
        raise EditorError('empty_input', 'Input is missing or empty.')
    if path.stat().st_size > limits.max_upload_bytes:
        raise EditorError('upload_too_large', f'Upload exceeds {limits.max_upload_bytes // 2**20} MiB.')
    if user_upload and path.suffix.lower() not in ALLOWED_SUFFIXES:
        raise EditorError('unsupported_format', 'Use WAV, MP3, FLAC, M4A, MP4, MOV, OGG, Opus, or WebM.')
    raw = command(['ffprobe', '-v', 'error', '-protocol_whitelist', 'file,pipe',
                   '-format_whitelist', FORMAT_WHITELIST, '-probesize', '5000000',
                   '-analyzeduration', '5000000', '-show_streams', '-show_format', '-of', 'json', str(path)],
                  timeout=limits.ffmpeg_timeout)
    try:
        data = json.loads(raw)
        streams = [s for s in data.get('streams', []) if s.get('codec_type') == 'audio']
        if len(streams) != 1:
            raise EditorError('audio_stream_count', 'Provide exactly one audio track; preselect the desired track for multitrack video.')
        stream = streams[0]
        duration = stream.get('duration', data.get('format', {}).get('duration'))
        if duration in (None, 'N/A'):
            duration = data.get('format', {}).get('duration')
        duration = require_finite(duration, 'media duration', limits.min_audio_seconds, limits.max_source_seconds)
        channels = int(stream['channels'])
        sample_rate = int(stream['sample_rate'])
        if channels not in (1, 2) or not 8000 <= sample_rate <= 192000:
            raise EditorError('unsupported_audio', 'Only mono/stereo audio at 8-192 kHz is accepted.')
        return {'duration_seconds': duration, 'channels': channels, 'sample_rate': sample_rate,
                'has_video': any(s.get('codec_type') == 'video' for s in data['streams']),
                'bytes': path.stat().st_size}
    except (KeyError, ValueError, TypeError) as exc:
        raise EditorError('bad_media_metadata', 'Could not determine valid audio metadata.') from exc


def measure_signal(path: Path, limits: Limits) -> dict:
    raw = command(['ffmpeg', '-hide_banner', '-v', 'error', '-nostdin', '-xerror',
                   '-protocol_whitelist', 'file,pipe', '-format_whitelist', FORMAT_WHITELIST,
                   '-i', str(path), '-map', '0:a:0', '-vn', '-sn', '-dn',
                   '-t', str(limits.max_output_seconds + 1), '-ac', '1', '-ar', '24000',
                   '-f', 'f32le', '-c:a', 'pcm_f32le', 'pipe:1'],
                  timeout=limits.ffmpeg_timeout, binary=True)
    if len(raw) < 4 or len(raw) % 4:
        raise EditorError('bad_pcm', 'Decoded audio is empty or malformed.')
    samples = array.array('f'); samples.frombytes(raw)
    if sys.byteorder != 'little': samples.byteswap()
    if not all(math.isfinite(x) for x in samples):
        raise EditorError('nonfinite_audio', 'Output contains NaN or infinite samples.')
    peak = max(abs(x) for x in samples)
    rms = math.sqrt(math.fsum(x*x for x in samples) / len(samples))
    clip_fraction = sum(abs(x) >= .999 for x in samples) / len(samples)
    return {'frames': len(samples), 'duration_seconds': len(samples)/24000,
            'peak': peak, 'rms_dbfs': 20 * math.log10(max(rms, 1e-12)),
            'near_full_scale_fraction': clip_fraction}


def prepare_audio(source: Path, out: Path, start: float, end: float | None,
                  limits: Limits, channel: str = 'mix') -> dict:
    meta = probe_audio(source, limits)
    start = require_finite(start, 'start_seconds', 0, meta['duration_seconds'])
    end = meta['duration_seconds'] if end is None else require_finite(end, 'end_seconds', 0, meta['duration_seconds'])
    span = end - start
    require_finite(span, 'selected duration', limits.min_audio_seconds, limits.max_input_seconds)
    if channel not in {'mix', 'left', 'right'}:
        raise EditorError('invalid_channel', 'Channel must be mix, left, or right.')
    if meta['channels'] == 1 and channel == 'right':
        raise EditorError('invalid_channel', 'A mono recording does not have a right channel.')
    args = ['ffmpeg', '-hide_banner', '-v', 'error', '-nostdin', '-xerror',
            '-protocol_whitelist', 'file,pipe', '-format_whitelist', FORMAT_WHITELIST,
            '-i', str(source), '-map', '0:a:0', '-vn', '-sn', '-dn',
            '-ss', f'{start:.6f}', '-t', f'{span:.6f}']
    if channel != 'mix':
        args += ['-af', f'pan=mono|c0=c{0 if channel == "left" else 1}']
    args += ['-ac', '1', '-ar', '24000', '-c:a', 'pcm_s16le', '-f', 'wav', '-y', str(out)]
    command(args, timeout=limits.ffmpeg_timeout)
    signal_stats = measure_signal(out, limits)
    if abs(signal_stats['duration_seconds'] - span) > .06:
        raise EditorError('truncated_input', 'Decoded input is shorter than the selected range.')
    if signal_stats['rms_dbfs'] <= -100:
        raise EditorError('silent_input', 'Selected audio is effectively silent. Check the channel/range.')
    warnings = []
    if signal_stats['rms_dbfs'] < -55: warnings.append('Input is very quiet; speech quality is not established.')
    if signal_stats['near_full_scale_fraction'] > .005: warnings.append('Input may already be clipped.')
    if meta['channels'] == 2 and channel == 'mix': warnings.append('Stereo was explicitly downmixed; output is mono.')
    if meta['has_video']: warnings.append('Only the voice track is exported; no video remux or lip-sync guarantee.')
    if start > 0 or end < meta['duration_seconds'] - .02:
        warnings.append('Output is only the selected clip; it is NOT spliced into the original recording.')
    return {'source': meta, 'start_seconds': start, 'end_seconds': end, 'channel': channel,
            'signal': signal_stats, 'warnings': warnings}


def validate_output(path: Path, target: float, limits: Limits) -> dict:
    meta = probe_audio(path, limits, user_upload=False)
    if meta['sample_rate'] != 24000 or meta['channels'] != 1:
        raise EditorError('wrong_output_format', 'Expected a mono 24 kHz output from AuK.')
    if abs(meta['duration_seconds'] - target) > .06:
        raise EditorError('wrong_output_duration', 'Output duration does not match the requested target.')
    stats = measure_signal(path, limits)
    if abs(stats['duration_seconds'] - target) > .06:
        raise EditorError('truncated_output', 'Decoded output is incomplete.')
    if stats['rms_dbfs'] <= -100:
        raise EditorError('silent_output', 'Model returned effectively silent audio.')
    if stats['peak'] > 1.001:
        raise EditorError('out_of_range_output', 'Audio exceeds full scale; output was not silently normalized.')
    warnings = []
    if stats['near_full_scale_fraction'] > .005: warnings.append('Output may contain clipping; listen before approving.')
    if stats['rms_dbfs'] < -55: warnings.append('Output is very quiet; inspect the recording.')
    return {'technical_checks': 'passed', 'signal': stats, 'warnings': warnings,
            'naturalness': 'not_automatically_verified', 'speaker_identity': 'not_automatically_verified'}


def decode_response(response: dict, limits: Limits) -> bytes:
    if not isinstance(response, dict) or not isinstance(response.get('audio'), str):
        raise EditorError('missing_audio', 'Server returned no base64 audio field.')
    encoded = response['audio']
    if len(encoded) > limits.response_bytes:
        raise EditorError('response_too_large', 'Audio response exceeded the configured bound.')
    try:
        content = base64.b64decode(encoded, validate=True)
    except Exception as exc:
        raise EditorError('invalid_base64', 'Server audio is not valid base64.') from exc
    if len(content) < 44 or content[:4] != b'RIFF' or content[8:12] != b'WAVE':
        raise EditorError('invalid_wav', 'Server did not return a RIFF/WAVE file.')
    return content


def normalized_tokens(text: str) -> list[str]:
    # A documented heuristic, not a multilingual semantic equivalence metric.
    text = unicodedata.normalize('NFKC', text).casefold()
    return re.findall(r"[^\W_]+(?:['’][^\W_]+)?", text, flags=re.UNICODE)


def edit_distance(a: list, b: list) -> int:
    previous = list(range(len(b) + 1))
    for i, left in enumerate(a, 1):
        row = [i]
        for j, right in enumerate(b, 1):
            row.append(min(row[-1]+1, previous[j]+1, previous[j-1]+(left != right)))
        previous = row
    return previous[-1]


def compare_transcript(expected: str, observed: str) -> dict:
    a, b = normalized_tokens(expected), normalized_tokens(observed)
    if not a:
        raise EditorError('empty_expected_text', 'Expected transcript has no comparable characters.')
    wer = edit_distance(a, b) / len(a)
    # Use character comparison for CJK text; English word-error rate is not suitable there.
    is_cjk = any('\u3400' <= c <= '\u9fff' for c in expected)
    if is_cjk:
        a, b = list(''.join(a)), list(''.join(b))
    error = edit_distance(a, b) / max(1, len(a))
    return {'metric': 'character_error_rate' if is_cjk else 'word_error_rate',
            'error_rate': round(error, 4), 'word_error_rate': round(wer, 4),
            'exact_normalized_match': a == b,
            'status': 'no_text_discrepancy_detected' if a == b else 'review_required'}


In [ ]:
# @title Local server lifecycle and hard deadlines
# --- Local server lifecycle: single owner, no public tunnel, kill on deadline ---

def process_start_ticks(pid: int) -> str | None:
    try:
        return Path(f'/proc/{pid}/stat').read_text().rsplit(')', 1)[1].split()[19]
    except (OSError, IndexError):
        return None


def kill_owned_group(pid: int, ticks: str | None) -> None:
    if not ticks or process_start_ticks(pid) != ticks:
        return
    with contextlib.suppress(ProcessLookupError): os.killpg(pid, signal.SIGTERM)
    deadline = time.monotonic() + 4
    while process_start_ticks(pid) == ticks and time.monotonic() < deadline:
        # Zombies have exited and cannot retain GPU memory.
        try:
            if Path(f'/proc/{pid}/stat').read_text().rsplit(')', 1)[1].split()[0] == 'Z': return
        except OSError: return
        time.sleep(.05)
    if process_start_ticks(pid) == ticks:
        with contextlib.suppress(ProcessLookupError): os.killpg(pid, signal.SIGKILL)


class LocalAuKServer:
    def __init__(self, binary: Path, model_dir: Path, root: Path, limits: Limits, memory_saver: bool):
        self.binary, self.model_dir, self.root = Path(binary).resolve(), Path(model_dir).resolve(), Path(root).resolve()
        self.limits, self.memory_saver = limits, bool(memory_saver)
        self.root.mkdir(parents=True, exist_ok=True)
        self.owner_file = self.root / 'server_owner.json'
        self.process = None; self.log_handle = None; self.profile = None; self.has_run = False
        self.model_id = 'auk-voice-' + uuid.uuid4().hex
        self.url = ''
        self.recover_orphan()

    def recover_orphan(self) -> None:
        if not self.owner_file.exists(): return
        info = json.loads(self.owner_file.read_text())
        pid = int(info['pid'])
        # The Editor's process lock is already held. Do not kill unrelated processes.
        try:
            argv = Path(f'/proc/{pid}/cmdline').read_bytes().split(b'\0')
            if str(info['config']).encode() in argv:
                kill_owned_group(pid, info.get('start_ticks'))
        except OSError: pass
        self.owner_file.unlink(missing_ok=True)

    def stop(self) -> None:
        if self.process is not None:
            kill_owned_group(self.process.pid, self.start_ticks)
            with contextlib.suppress(subprocess.TimeoutExpired): self.process.wait(timeout=5)
        self.process = None
        if self.log_handle is not None: self.log_handle.close()
        self.log_handle = None; self.profile = None; self.has_run = False
        self.owner_file.unlink(missing_ok=True)

    def _get(self, route: str) -> dict:
        opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
        with opener.open(self.url + route, timeout=2) as response:
            return json.loads(response.read(100_000))

    def start(self, profile: str) -> bool:
        if profile not in PROFILE_FILES:
            raise EditorError('unknown_profile', 'Select a supported Base/Flash profile.')
        if self.profile == profile and self.process is not None and self.process.poll() is None:
            return False
        self.stop()
        variant, generator = PROFILE_FILES[profile]
        for name in (generator, *SHARED_FILES, *SIDECARS):
            if not (self.model_dir / name).is_file():
                raise EditorError('missing_model', f'Missing {name}. Rerun the model-download cell for {profile}.')
        with socket.socket() as sock:
            sock.bind(('127.0.0.1', 0)); port = sock.getsockname()[1]
        self.url = f'http://127.0.0.1:{port}'
        config = {
            'host': '127.0.0.1', 'port': port, 'backend': 'cuda', 'device': 0,
            'threads': 2, 'lazy_load': False, 'max_loaded_models': 1,
            'log_request_body': False, 'max_request_body_bytes': 64 * 1024,
            'models': [{
                'id': self.model_id, 'family': 'auk', 'path': str(self.model_dir),
                'task': 'gen', 'mode': 'offline',
                'session_options': {
                    'auk.variant': variant, 'auk.model_gguf': generator,
                    'auk.qwen_gguf': SHARED_FILES[0], 'auk.vae_gguf': SHARED_FILES[1],
                    'auk.mem_saver': str(self.memory_saver).lower(),
                },
                'default_request_options': {
                    'num_inference_steps': 4 if variant == 'flash' else 32,
                    'guidance_scale': 0.0 if variant == 'flash' else 2.0,
                    'sway_sampling_coef': -1.0,
                },
            }],
        }
        specs = self.binary.parent / 'model_specs' / 'auk.json'
        if specs.is_file(): config['model_spec_override'] = str(specs)
        config_path = self.root / ('server-' + self.model_id + '.json')
        atomic_json(config_path, config)
        log_path = self.root / 'server.log'
        # Bound retained logs: previous startup only; application does not enable --log.
        if log_path.exists(): os.replace(log_path, self.root / 'server.previous.log')
        self.log_handle = log_path.open('wb')
        try:
            self.process = subprocess.Popen([str(self.binary), '--config', str(config_path), '--no-ui'],
                cwd=self.binary.parent, stdin=subprocess.DEVNULL, stdout=self.log_handle,
                stderr=subprocess.STDOUT, start_new_session=True)
            self.start_ticks = process_start_ticks(self.process.pid)
            atomic_json(self.owner_file, {'pid': self.process.pid, 'start_ticks': self.start_ticks,
                                         'config': str(config_path)})
            deadline = time.monotonic() + self.limits.startup_timeout
            while time.monotonic() < deadline:
                if self.process.poll() is not None:
                    raise EditorError('server_start_failed', 'AuK server exited at startup. Inspect the private server log; check driver, model hashes, and memory.')
                try:
                    self._get('/health')
                    models = self._get('/v1/models')
                    if any(item.get('id') == self.model_id for item in models.get('data', [])):
                        self.profile = profile; self.has_run = False
                        return True
                except (OSError, ValueError, KeyError): pass
                time.sleep(.4)
            raise EditorError('server_start_timeout', 'Server exceeded its startup deadline; its process was stopped.')
        except BaseException:
            self.stop()
            raise

    def run(self, profile: str, input_path: Path, plan: dict) -> tuple[dict, dict]:
        started = time.monotonic()
        self.start(profile)
        startup_seconds = time.monotonic() - started
        cold = not self.has_run
        payload = {'model': self.model_id, 'request': {
            'audio': str(Path(input_path).resolve()), 'text': plan['instruction'],
            'seed': plan['seed'], 'options': {'duration_sec': plan['target_seconds']}}}
        proc, ticks = self.process, self.start_ticks
        expired = threading.Event()
        def watchdog():
            expired.set()
            kill_owned_group(proc.pid, ticks)
        timer = threading.Timer(self.limits.inference_timeout, watchdog)
        timer.daemon = True
        call_started = time.monotonic(); timer.start()
        success = False
        try:
            opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
            request = urllib.request.Request(self.url + '/v1/tasks/run',
                data=json.dumps(payload, ensure_ascii=False, allow_nan=False).encode(),
                headers={'Content-Type': 'application/json'}, method='POST')
            with opener.open(request, timeout=self.limits.inference_timeout) as response:
                body = response.read(self.limits.response_bytes + 1)
            if len(body) > self.limits.response_bytes:
                raise EditorError('response_too_large', 'Response exceeded its size bound.')
            if expired.is_set(): raise EditorError('inference_timeout', 'Inference deadline exceeded; worker process stopped.')
            parsed = json.loads(body)
            if not isinstance(parsed, dict): raise EditorError('invalid_response', 'Server returned an invalid response object.')
            success = True; self.has_run = True
            return parsed, {'server_start_seconds': round(startup_seconds, 3),
                            'inference_seconds': round(time.monotonic() - call_started, 3),
                            'first_request_after_start': cold,
                            'memory_saver': self.memory_saver}
        except urllib.error.HTTPError as exc:
            detail = exc.read(16_384).decode(errors='replace').lower()
            oom = any(x in detail for x in ('out of memory', 'cuda_error_out_of_memory', 'insufficient_memory', 'cuda allocation'))
            code = 'out_of_memory' if oom else 'server_http_error'
            message = ('GPU memory was exhausted. Select memory-saving mode or a smaller profile, then explicitly retry.'
                       if oom else f'AuK returned HTTP {exc.code}. Inspect the private server log. No fallback output was substituted.')
            raise EditorError(code, message) from exc
        except (OSError, ValueError) as exc:
            code = 'inference_timeout' if expired.is_set() else 'inference_transport_error'
            raise EditorError(code, 'Inference did not complete; its server process was stopped. An explicit retry is required.') from exc
        finally:
            timer.cancel(); timer.join(timeout=6)
            if not success: self.stop()


In [ ]:
# @title Restart-safe jobs, cache integrity, and explicit review
# --- Persistent single-user jobs, idempotency, explicit review, no billing claims ---

class VoiceEditor:
    def __init__(self, root: Path, binary: Path, model_dir: Path, *, memory_saver: bool,
                 provenance: dict, limits: Limits = Limits(), commercial: bool = False,
                 commercial_license_confirmed: bool = False,
                 transport_factory: Callable | None = None, asr: Callable | None = None,
                 asr_fingerprint: str = 'disabled'):
        if commercial and not commercial_license_confirmed:
            raise EditorError('license_not_cleared', 'Commercial use is disabled until the required model licenses are independently cleared.')
        self.root = Path(root).resolve(); self.root.mkdir(parents=True, exist_ok=True)
        os.chmod(self.root, 0o700)
        self.limits, self.provenance = limits, dict(provenance)
        self.provenance.update({'core_version': CORE_VERSION, 'memory_saver': bool(memory_saver),
                                'asr_fingerprint': asr_fingerprint})
        self.asr = asr
        self._mutex = threading.Lock()
        self._lockfile = (self.root / '.worker.lock').open('a+')
        try:
            fcntl.flock(self._lockfile, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            self._lockfile.close()
            raise EditorError('worker_busy', 'Another editor owns this job directory. Stop it before creating another.') from exc
        try:
            self.db = sqlite3.connect(self.root / 'jobs.sqlite3')
            self.db.execute('PRAGMA journal_mode=WAL'); self.db.execute('PRAGMA synchronous=FULL')
            self.db.execute('CREATE TABLE IF NOT EXISTS jobs (job_id TEXT PRIMARY KEY, fingerprint TEXT UNIQUE NOT NULL, status TEXT NOT NULL, updated_at TEXT NOT NULL)')
            self.db.commit()
            factory = transport_factory or LocalAuKServer
            self.transport = factory(binary, model_dir, self.root / 'runtime', limits, memory_saver)
            # No in-flight job can remain valid after ownership is reacquired.
            stale = self.db.execute("SELECT job_id FROM jobs WHERE status IN ('queued','running')").fetchall()
            for (job_id,) in stale:
                p = self.root / 'jobs' / job_id / 'manifest.json'
                old = json.loads(p.read_text()) if p.is_file() else {'job_id': job_id}
                old.update({'status': 'interrupted', 'error_code': 'worker_restarted', 'updated_at': utc_now()})
                atomic_json(p, old)
            self.db.execute("UPDATE jobs SET status='interrupted', updated_at=? WHERE status IN ('queued','running')", (utc_now(),))
            self.db.commit()
        except BaseException:
            with contextlib.suppress(Exception): self.db.close()
            fcntl.flock(self._lockfile, fcntl.LOCK_UN); self._lockfile.close()
            raise

    def close(self) -> None:
        if self._lockfile.closed: return
        self.transport.stop(); self.db.close()
        fcntl.flock(self._lockfile, fcntl.LOCK_UN); self._lockfile.close()

    def _save(self, job_dir: Path, manifest: dict, status: str) -> None:
        manifest['status'] = status; manifest['updated_at'] = utc_now()
        atomic_json(job_dir / 'manifest.json', manifest)
        self.db.execute('UPDATE jobs SET status=?, updated_at=? WHERE job_id=?',
                        (status, manifest['updated_at'], manifest['job_id']))
        self.db.commit()

    def edit(self, source: Path, spec: EditSpec, *, profile: str = 'base_f32',
             start_seconds: float = 0, end_seconds: float | None = None, channel: str = 'mix',
             rights_confirmed: bool = False, retry_failed: bool = False,
             allow_unchecked_word_edits: bool = False, cache_namespace: str = '') -> dict:
        if not self._mutex.acquire(blocking=False):
            raise EditorError('worker_busy', 'One edit is already running in this worker.')
        try:
            return self._edit(source, spec, profile=profile, start_seconds=start_seconds,
                              end_seconds=end_seconds, channel=channel, rights_confirmed=rights_confirmed,
                              retry_failed=retry_failed, allow_unchecked_word_edits=allow_unchecked_word_edits,
                              cache_namespace=cache_namespace)
        finally:
            self._mutex.release()

    def _edit(self, source: Path, spec: EditSpec, **options) -> dict:
        if not options['rights_confirmed']:
            raise EditorError('rights_not_confirmed', 'Confirm you are authorized to edit this recording before running it.')
        profile = options['profile']
        if profile not in PROFILE_FILES: raise EditorError('unknown_profile', 'Choose a supported profile.')
        source = Path(source).resolve()
        probe_audio(source, self.limits)  # Fail before hashing/staging a very large/unsupported file.
        if spec.operation == 'replace_words' and self.asr is None and not options['allow_unchecked_word_edits']:
            raise EditorError('asr_required', 'Enable optional ASR QA for word edits, or explicitly allow an unchecked evaluation output.')
        identity = {'source_sha256': sha256_file(source), 'spec': dataclasses.asdict(spec),
                    'profile': profile, 'start': options['start_seconds'], 'end': options['end_seconds'],
                    'channel': options['channel'], 'limits': dataclasses.asdict(self.limits),
                    'provenance': self.provenance, 'cache_namespace': options['cache_namespace']}
        fingerprint = canonical_hash(identity)
        row = self.db.execute('SELECT job_id, status FROM jobs WHERE fingerprint=?', (fingerprint,)).fetchone()
        if row:
            job_id, state = row
            job_dir = self.root / 'jobs' / job_id
            manifest = json.loads((job_dir / 'manifest.json').read_text())
            output = job_dir / 'output.wav'
            if state == 'generated':
                if not output.is_file() or sha256_file(output) != manifest.get('output_sha256'):
                    raise EditorError('corrupt_cached_output', 'Cached output is missing or changed; it was not returned as a valid result.')
                return {**manifest, 'cache_hit': True, 'output_path': str(output), 'job_dir': str(job_dir)}
            if not options['retry_failed']:
                raise EditorError('explicit_retry_required', f'Previous attempt is {state}. Enable retry_failed to rerun the same job.')
        else:
            job_id = uuid.uuid4().hex
            job_dir = self.root / 'jobs' / job_id; job_dir.mkdir(parents=True)
            manifest = {'job_id': job_id, 'fingerprint': fingerprint, 'created_at': utc_now(), 'attempts': 0}
            self.db.execute('INSERT INTO jobs VALUES (?,?,?,?)', (job_id, fingerprint, 'queued', utc_now())); self.db.commit()
        manifest.update({'request': identity, 'attempts': manifest.get('attempts', 0)+1,
                         'human_review': 'pending', 'semantic_validation': 'not_run',
                         'rights_confirmed': True, 'error_code': None})
        output = job_dir / 'output.wav'; output.unlink(missing_ok=True)
        self._save(job_dir, manifest, 'running')
        started = time.monotonic()
        try:
            prepared = job_dir / 'input.wav'
            media = prepare_audio(source, prepared, options['start_seconds'], options['end_seconds'],
                                  self.limits, options['channel'])
            # Source can be a user-managed local path. Detect a change while it was read.
            if sha256_file(source) != identity['source_sha256']:
                raise EditorError('source_changed', 'Source changed during preparation; upload a stable copy and retry.')
            plan = build_plan(spec, media['signal']['duration_seconds'], self.limits)
            manifest['input'] = media; manifest['plan'] = plan
            self._save(job_dir, manifest, 'running')
            qa_started = time.monotonic()
            source_asr = None
            if self.asr is not None and (plan['preserve_content'] or spec.operation == 'replace_words'):
                source_asr = self.asr(prepared)
                if not source_asr.get('text', '').strip():
                    raise EditorError('speech_not_detected', 'ASR did not detect words. Check the clip; this may be an ASR failure.')
                if spec.operation == 'replace_words':
                    source_check = compare_transcript(spec.source_transcript, source_asr['text'])
                    if source_check['error_rate'] > .2:
                        raise EditorError('source_transcript_mismatch', 'ASR substantially disagrees with the supplied transcript; review the clip/transcript before replacing words.')
            manifest['source_asr'] = source_asr
            pre_qa_seconds = time.monotonic() - qa_started
            response, timing = self.transport.run(profile, prepared, plan)
            raw_path = job_dir / 'raw-output.wav'
            atomic_bytes(raw_path, decode_response(response, self.limits))
            quality = validate_output(raw_path, plan['target_seconds'], self.limits)
            # 16-bit WAV export, no loudness normalization, denoiser, TTS, or synthetic fallback.
            temporary = job_dir / 'export.tmp.wav'
            command(['ffmpeg', '-hide_banner', '-v', 'error', '-nostdin', '-xerror', '-i', str(raw_path),
                     '-map', '0:a:0', '-c:a', 'pcm_s16le', '-f', 'wav', '-y', str(temporary)],
                    timeout=self.limits.ffmpeg_timeout)
            validate_output(temporary, plan['target_seconds'], self.limits)
            os.replace(temporary, output)
            raw_path.unlink(missing_ok=True)
            expected = plan['expected_transcript']
            if not expected and source_asr and plan['preserve_content']:
                expected = source_asr['text']
            qa_started = time.monotonic()
            if self.asr is not None and expected:
                output_asr = self.asr(output)
                quality['transcript_check'] = compare_transcript(expected, output_asr.get('text', ''))
                quality['output_asr'] = output_asr
                if spec.operation == 'replace_words':
                    new_tokens = normalized_tokens(spec.new_words)
                    actual = normalized_tokens(output_asr.get('text', ''))
                    present = bool(new_tokens) and any(actual[i:i+len(new_tokens)] == new_tokens for i in range(len(actual)))
                    quality['replacement_detected'] = present
                    if not present: quality['transcript_check']['status'] = 'review_required'
                manifest['semantic_validation'] = quality['transcript_check']['status']
                if manifest['semantic_validation'] == 'review_required':
                    quality['warnings'].append('ASR detected a text discrepancy. Do not treat this output as an approved edit.')
            else:
                manifest['semantic_validation'] = 'not_run'
                quality['warnings'].append('No automatic content check was run. A valid WAV is not proof that the requested edit succeeded.')
            timing['qa_seconds'] = round(pre_qa_seconds + time.monotonic() - qa_started, 3)
            timing['total_seconds'] = round(time.monotonic() - started, 3)
            timing['inference_real_time_factor'] = round(timing['inference_seconds']/plan['target_seconds'], 3)
            manifest.update({'quality': quality, 'timing': timing, 'output_sha256': sha256_file(output),
                             'input_sha256': sha256_file(prepared), 'profile_used': profile,
                             'warnings': media['warnings'] + quality['warnings']})
            self._save(job_dir, manifest, 'generated')
            return {**manifest, 'cache_hit': False, 'output_path': str(output), 'job_dir': str(job_dir)}
        except BaseException as exc:
            # Never leave a stale playable output beside a failed manifest.
            for name in ('output.wav', 'raw-output.wav', 'export.tmp.wav'):
                (job_dir / name).unlink(missing_ok=True)
            manifest['error_code'] = getattr(exc, 'code', type(exc).__name__)
            manifest['elapsed_before_failure_seconds'] = round(time.monotonic() - started, 3)
            self._save(job_dir, manifest, 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed')
            raise

    def review(self, job_id: str, *, approved: bool, notes: str = '') -> dict:
        if not re.fullmatch(r'[a-f0-9]{32}', job_id): raise EditorError('invalid_job_id', 'Invalid job ID.')
        job_dir = self.root / 'jobs' / job_id
        row = self.db.execute('SELECT status FROM jobs WHERE job_id=?', (job_id,)).fetchone()
        if not row or row[0] != 'generated': raise EditorError('not_reviewable', 'Only a generated output can be reviewed.')
        manifest = json.loads((job_dir / 'manifest.json').read_text())
        if approved and (not (job_dir/'output.wav').is_file() or sha256_file(job_dir/'output.wav') != manifest['output_sha256']):
            raise EditorError('corrupt_output', 'Output changed since generation; approval blocked.')
        manifest.update({'human_review': 'approved' if approved else 'rejected',
                         'review_notes': str(notes)[:2000], 'reviewed_at': utc_now()})
        self._save(job_dir, manifest, 'generated')
        return manifest


In [ ]:
# @title Artifact download and checksum helpers
# Setup helpers: download immutable artifacts, check hashes, and keep a lock manifest.
def download_file(url: str, destination: Path, *, expected_sha256: str | None = None,
                  max_seconds: int = 1800) -> Path:
    if urllib.parse.urlparse(url).scheme != 'https':
        raise EditorError('insecure_download', 'Artifact downloads require HTTPS.')
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and expected_sha256 and sha256_file(destination) == expected_sha256:
        return destination
    partial = destination.with_name(destination.name + '.part')
    # curl can resume a large HF download. A mismatched final digest is never accepted.
    args = ['curl', '--fail', '--location', '--show-error', '--silent', '--proto', '=https',
            '--proto-redir', '=https', '--connect-timeout', '20', '--max-time', str(max_seconds),
            '--retry', '3', '--retry-delay', '2', '--retry-all-errors',
            '--continue-at', '-', '--output', str(partial), url]
    command(args, timeout=max_seconds * 4 + 30)
    if not partial.is_file() or partial.stat().st_size == 0:
        raise EditorError('download_empty', f'Download returned no bytes: {destination.name}')
    if expected_sha256 and sha256_file(partial) != expected_sha256:
        partial.unlink(missing_ok=True)
        raise EditorError('checksum_mismatch', f'Checksum failed for {destination.name}; corrupt partial removed. Rerun setup.')
    os.replace(partial, destination)
    return destination


def fetch_text(url: str, *, limit: int = 2 * 1024 * 1024) -> str:
    req = urllib.request.Request(url, headers={'User-Agent': 'MagicHour-AuK-Colab/1.0'})
    last = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=30) as response:
                data = response.read(limit + 1)
            if len(data) > limit: raise EditorError('metadata_too_large', 'Metadata exceeded its size bound.')
            return data.decode('utf-8')
        except (urllib.error.URLError, TimeoutError) as exc:
            last = exc
            if attempt < 2: time.sleep(2 ** attempt)
    raise EditorError('metadata_unavailable', 'Could not fetch pinned artifact metadata. Check network access, then rerun setup.') from last


def parse_lfs_pointer(text: str) -> tuple[str, int]:
    digest = re.search(r'^oid sha256:([a-f0-9]{64})$', text, flags=re.MULTILINE)
    size = re.search(r'^size ([0-9]+)$', text, flags=re.MULTILINE)
    if not digest or not size:
        raise EditorError('bad_model_pointer', 'Expected a SHA-256 Git LFS pointer from the pinned model revision.')
    count = int(size.group(1))
    if count <= 0: raise EditorError('bad_model_pointer', 'Model pointer has invalid size.')
    return digest.group(1), count


def safe_extract(archive_path: Path, target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        if sum(m.size for m in members) > 4 * 2**30:
            raise EditorError('archive_too_large', 'Runtime archive expands beyond the safety bound.')
        for member in members:
            p = (root / member.name).resolve()
            if p != root and root not in p.parents:
                raise EditorError('unsafe_archive', 'Archive path escapes its destination.')
            if not (member.isfile() or member.isdir()):
                raise EditorError('unsafe_archive', 'Links/devices in runtime archives are not accepted.')
        for member in members:
            archive.extract(member, root, filter='data')


def gpu_details() -> dict:
    row = command(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,compute_cap',
                   '--format=csv,noheader,nounits'], timeout=10).strip().splitlines()[0]
    name, total, free, capability = [part.strip() for part in row.split(',')]
    return {'name': name, 'total_mib': int(total), 'free_mib': int(free), 'compute_capability': capability}


def setup_runtime(root: Path, *, mode: str, gpu: dict) -> tuple[Path, dict]:
    target = root / 'runtime-package'
    lock_path = target / 'runtime.lock.json'
    if mode not in {'t4_prebuilt', 'source_build'}: raise EditorError('bad_setup_mode', 'Select t4_prebuilt or source_build.')
    if mode == 't4_prebuilt' and gpu['compute_capability'] != '7.5':
        raise EditorError('wrong_gpu', 'This prebuilt targets T4/SM75. Select a T4, or choose source_build explicitly for the attached GPU.')
    if lock_path.is_file():
        lock = json.loads(lock_path.read_text())
        binary = Path(lock['binary'])
        if (lock.get('mode') == mode and lock.get('tag') == RUNTIME_TAG
            and lock.get('compute_capability') == gpu['compute_capability']
            and binary.is_file() and sha256_file(binary) == lock.get('binary_sha256')):
            command([str(binary), '--help'], timeout=20)
            return binary, lock
    target.mkdir(parents=True, exist_ok=True)
    if mode == 't4_prebuilt':
        archive = root / RUNTIME_ASSET
        print('Downloading checksum-pinned T4 runtime...')
        download_file(RUNTIME_URL, archive, expected_sha256=RUNTIME_SHA256, max_seconds=300)
        unpacked = target / 'unpacked'
        if unpacked.exists(): shutil.rmtree(unpacked)
        safe_extract(archive, unpacked)
        candidates = [p for p in unpacked.rglob('audiocpp_server') if p.is_file()]
        if len(candidates) != 1: raise EditorError('bad_runtime_archive', 'Expected exactly one server binary.')
        binary = candidates[0]; binary.chmod(0o755)
        extra = {'archive_sha256': RUNTIME_SHA256}
    else:
        if not shutil.which('nvcc'): raise EditorError('cuda_toolkit_missing', 'source_build requires the CUDA development toolkit (nvcc).')
        source = root / 'audio-cpp-source'
        if not (source / '.git').is_dir():
            command(['git', 'clone', '--filter=blob:none', '--no-checkout',
                     'https://github.com/0xShug0/audio.cpp.git', str(source)], timeout=600)
        command(['git', 'fetch', '--depth', '1', 'origin', 'tag', RUNTIME_TAG], cwd=source, timeout=300)
        revision = command(['git', 'rev-parse', f'{RUNTIME_TAG}^{{commit}}'], cwd=source).strip()
        if not re.fullmatch(r'[a-f0-9]{40}', revision) or not revision.startswith(RUNTIME_COMMIT_PREFIX):
            raise EditorError('unexpected_source_commit', 'Release source no longer resolves to the verified commit prefix. Stop and review it.')
        command(['git', 'checkout', '--detach', revision], cwd=source)
        command(['git', '-c', 'url.https://github.com/.insteadOf=git@github.com:',
                 'submodule', 'update', '--init', '--recursive'], cwd=source, timeout=600)
        architecture = gpu['compute_capability'].replace('.', '')
        build = source / ('build-colab-sm' + architecture)
        command(['cmake', '-S', str(source), '-B', str(build), '-DCMAKE_BUILD_TYPE=Release',
                 '-DENGINE_ENABLE_CUDA=ON', '-DENGINE_ENABLE_VULKAN=OFF',
                 '-DENGINE_ENABLE_NATIVE_CPU=OFF', '-DENGINE_BUILD_TESTS=OFF',
                 '-DAUDIOCPP_DEPLOYMENT_BUILD=ON', '-DAUDIOCPP_BUILD_NATIVE_MODEL_MANAGER=OFF',
                 f'-DCMAKE_CUDA_ARCHITECTURES={architecture}'], timeout=600)
        command(['cmake', '--build', str(build), '--parallel', '2', '--target', 'audiocpp_server'], timeout=3600)
        candidates = [p for p in build.rglob('audiocpp_server') if p.is_file()]
        if len(candidates) != 1: raise EditorError('build_missing_binary', 'Build did not produce one server executable.')
        binary = candidates[0]
        extra = {'source_commit': revision}
    command([str(binary), '--help'], timeout=30)
    lock = {'tag': RUNTIME_TAG, 'mode': mode, 'binary': str(binary.resolve()),
            'binary_sha256': sha256_file(binary), 'compute_capability': gpu['compute_capability'],
            'created_at': utc_now(), **extra}
    atomic_json(lock_path, lock)
    return binary.resolve(), lock


def setup_models(root: Path, profiles: list[str]) -> tuple[Path, dict]:
    if not profiles or any(p not in PROFILE_FILES for p in profiles):
        raise EditorError('unknown_profile', 'Choose supported download profiles.')
    directory = root / 'models' / MODEL_REVISION
    directory.mkdir(parents=True, exist_ok=True)
    filenames = sorted({PROFILE_FILES[p][1] for p in profiles} | set(SHARED_FILES))
    remote = f'https://huggingface.co/{MODEL_REPO}'
    specs = {}
    for name in filenames:
        pointer = fetch_text(f'{remote}/raw/{MODEL_REVISION}/{name}', limit=16_384)
        digest, size = parse_lfs_pointer(pointer)
        specs[name] = {'sha256': digest, 'bytes': size}
    missing_size = sum(item['bytes'] for name, item in specs.items() if not (directory/name).is_file())
    if shutil.disk_usage(root).free < missing_size + 2 * 2**30:
        raise EditorError('disk_space', 'Insufficient disk space for the selected weights plus 2 GiB working space.')
    for name, spec in specs.items():
        print(f'Checking/downloading {name} ({spec["bytes"]/2**30:.2f} GiB)...')
        path = download_file(f'{remote}/resolve/{MODEL_REVISION}/{name}', directory/name,
                             expected_sha256=spec['sha256'])
        if path.stat().st_size != spec['bytes']:
            raise EditorError('model_size_mismatch', f'Model size is incorrect: {name}')
    for name in SIDECARS:
        # Immutable model commit fixes sidecar contents; record their hashes as well.
        path = directory / name
        if not path.is_file(): download_file(f'{remote}/resolve/{MODEL_REVISION}/{name}', path, max_seconds=300)
        if path.suffix == '.json': json.loads(path.read_text())
        specs[name] = {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
    lock = {'repo': MODEL_REPO, 'revision': MODEL_REVISION, 'files': specs}
    atomic_json(directory / 'model.lock.json', lock)
    return directory, lock


In [ ]:
# @title Optional isolated CPU speech QA helpers
# Optional CPU-only speech QA. Its isolated environment never imports Torch into this notebook.
ASR_MODEL_REPO = 'Systran/faster-whisper-small'
ASR_MODEL_REVISION = '536b0662742c02347bc0e980a01041f333bce120'
ASR_SCRIPT = r'''
import json, os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['HF_HUB_OFFLINE'] = '1'
from faster_whisper import WhisperModel
model_path, audio_path, output_path, language = sys.argv[1:]
model = WhisperModel(model_path, device='cpu', compute_type='int8', cpu_threads=2,
                     num_workers=1, local_files_only=True)
segments, info = model.transcribe(audio_path, beam_size=3, temperature=0,
    condition_on_previous_text=False, vad_filter=True, word_timestamps=True,
    language=None if language == 'auto' else language)
segments = list(segments)
result = {
    'text': ' '.join(s.text.strip() for s in segments).strip(),
    'language': info.language, 'language_probability': float(info.language_probability),
    'words': [{'word': w.word, 'start': w.start, 'end': w.end,
               'probability': float(w.probability)} for s in segments for w in (s.words or [])],
}
with open(output_path, 'w', encoding='utf-8') as stream:
    json.dump(result, stream, ensure_ascii=False, allow_nan=False)
'''


def setup_asr(root: Path, limits: Limits, language: str = 'auto') -> tuple[Callable, str]:
    if not re.fullmatch(r'auto|[a-z]{2,3}', language):
        raise EditorError('invalid_language', 'ASR language should be auto or an ISO language code such as en or zh.')
    qa_root = root / 'speech-qa'; qa_root.mkdir(parents=True, exist_ok=True)
    env = qa_root / 'venv'; py = env / 'bin/python'
    freeze_path = qa_root / 'requirements.lock.txt'
    if not py.is_file(): command([sys.executable, '-m', 'venv', str(env)], timeout=120)
    if not freeze_path.is_file():
        command([str(py), '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--quiet', 'faster-whisper==1.2.1'], timeout=900)
        frozen = command([str(py), '-m', 'pip', 'freeze'], timeout=30)
        atomic_bytes(freeze_path, frozen.encode())
    else:
        actual = command([str(py), '-m', 'pip', 'freeze'], timeout=30)
        if actual.strip() != freeze_path.read_text().strip():
            raise EditorError('qa_environment_changed', 'QA environment changed from its saved package lock. Recreate it explicitly before running.')
    command([str(py), '-m', 'pip', 'check'], timeout=30)
    model = qa_root / ASR_MODEL_REVISION; model.mkdir(exist_ok=True)
    remote = f'https://huggingface.co/{ASR_MODEL_REPO}'
    digest, size = parse_lfs_pointer(fetch_text(f'{remote}/raw/{ASR_MODEL_REVISION}/model.bin',limit=16_384))
    download_file(f'{remote}/resolve/{ASR_MODEL_REVISION}/model.bin', model/'model.bin', expected_sha256=digest)
    for name in ('config.json', 'tokenizer.json', 'vocabulary.txt'):
        if not (model/name).is_file(): download_file(f'{remote}/resolve/{ASR_MODEL_REVISION}/{name}',model/name,max_seconds=300)
    hashes = {p.name: sha256_file(p) for p in model.iterdir() if p.is_file()}
    script = qa_root / 'transcribe_local.py'; atomic_bytes(script, ASR_SCRIPT.encode())
    fingerprint = canonical_hash({'model': ASR_MODEL_REVISION, 'files': hashes,
                                  'requirements': freeze_path.read_text(), 'script': sha256_file(script),
                                  'language': language})
    atomic_json(qa_root/'qa.lock.json',{'revision':ASR_MODEL_REVISION,'fingerprint':fingerprint,
                                      'files':hashes,'language':language})
    def transcribe(path: Path) -> dict:
        fd, name = tempfile.mkstemp(suffix='.json', dir=qa_root); os.close(fd)
        out = Path(name)
        try:
            command([str(py), str(script), str(model), str(Path(path).resolve()), str(out), language],
                    timeout=limits.asr_timeout)
            value = json.loads(out.read_text())
            if not isinstance(value.get('text'),str): raise EditorError('invalid_asr_result','ASR did not return text.')
            return value
        finally:
            out.unlink(missing_ok=True)
    return transcribe, fingerprint


## 2. Install system tools and run local tests
The editing worker uses Python's standard library plus FFmpeg and audio.cpp. It does not install PyTorch, TorchAudio, xformers, or Gradio. Optional speech QA gets its own virtual environment later.

System packages use Colab's Ubuntu repositories; their installed versions are recorded, not claimed to be a fully immutable OS image. Production should pin a tested container digest. The 64 tests below use synthetic audio and a local fake inference server. They test the implementation, **not** AuK quality.

In [ ]:
# @title Install system tools
!apt-get -qq update && apt-get -qq install -y ca-certificates curl ffmpeg libgomp1 libssl-dev git cmake build-essential python3-venv
if get_ipython().user_ns.get("_exit_code", 0) != 0:
    raise RuntimeError("System package installation failed. Fix the error above before continuing.")
for executable in ("ffmpeg", "ffprobe", "curl"):
    if shutil.which(executable) is None:
        raise RuntimeError(f"Missing required tool: {executable}")
print(command(["ffmpeg", "-version"]).splitlines()[0])


In [ ]:
# @title Define the self-test suite (no GPU or model calls)
import io
import unittest
import wave
from unittest import mock


def make_test_wav(path, seconds=1.0, channels=1, amplitude=.2, rate=24000):
    frames = int(rate * seconds)
    samples = array.array('h')
    for i in range(frames):
        value = int(amplitude * 32767 * math.sin(2*math.pi*220*i/rate))
        samples.extend([value] * channels)
    if sys.byteorder != 'little': samples.byteswap()
    with wave.open(str(path), 'wb') as f:
        f.setnchannels(channels); f.setsampwidth(2); f.setframerate(rate); f.writeframes(samples.tobytes())
    return Path(path)


class FakeTransport:
    calls = 0
    fail = False
    def __init__(self, *args): self.has_run=False
    def stop(self): self.has_run=False
    def run(self, profile, input_path, plan):
        type(self).calls += 1
        if type(self).fail: raise EditorError('inference_timeout', 'Simulated timeout')
        # Fixtures test plumbing, not voice editing. Never used by the real UI.
        raw = Path(input_path).read_bytes()
        result = {'audio': base64.b64encode(raw).decode(), 'timing': {'fixture': True}}
        timing = {'inference_seconds': .1, 'server_start_seconds': 0,
                  'first_request_after_start': not self.has_run, 'memory_saver': True}
        self.has_run=True
        return result, timing


class PlannerTests(unittest.TestCase):
    def setUp(self): self.limits=Limits()
    def test_emotion_preserves_duration(self):
        p=build_plan(EditSpec(operation='emotion'),7,self.limits)
        self.assertEqual(p['target_seconds'],7); self.assertTrue(p['preserve_content'])
    def test_speed_sets_duration(self):
        p=build_plan(EditSpec(operation='speed',speed=1.5),9,self.limits)
        self.assertEqual(p['target_seconds'],6)
    def test_rounding_to_20ms(self):
        self.assertEqual(build_plan(EditSpec(instruction='Test',target_seconds=1.001),1,self.limits)['target_seconds'],1.02)
    def test_unknown_operation(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='whatever'),1,self.limits)
    def test_empty_custom(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(),1,self.limits)
    def test_invalid_emotion(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='emotion',emotion='n/a'),1,self.limits)
    def test_duration_budget(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='speed',speed=.75),15,self.limits)
    def test_nan_duration(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(instruction='Test'),float('nan'),self.limits)
    def test_invalid_seed(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(instruction='Test',seed=-1),1,self.limits)
    def test_boolean_seed(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(instruction='Test',seed=True),1,self.limits)
    def test_empty_voice_description(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='voice'),1,self.limits)
    def test_word_replacement(self):
        p=build_plan(EditSpec(operation='replace_words',source_transcript='Meet Tuesday.',old_words='Tuesday',new_words='Thursday'),2,self.limits)
        self.assertEqual(p['expected_transcript'],'Meet Thursday.')
    def test_ambiguous_word_replacement(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='replace_words',source_transcript='yes yes',old_words='yes',new_words='no'),2,self.limits)
    def test_replacement_needs_transcript(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='replace_words',old_words='yes',new_words='no'),2,self.limits)
    def test_conflicting_expected_text(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='emotion',source_transcript='Hello',expected_transcript='Goodbye'),2,self.limits)
    def test_nul_instruction(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(instruction='a\0b'),1,self.limits)
    def test_text_exact_match(self):
        self.assertEqual(compare_transcript('HELLO, world!', 'Hello world')['status'],'no_text_discrepancy_detected')
    def test_text_mismatch(self):
        self.assertEqual(compare_transcript('Tuesday', 'Thursday')['status'],'review_required')
    def test_cjk_metric(self):
        self.assertEqual(compare_transcript('你好世界', '你好')['metric'],'character_error_rate')
    def test_no_empty_expected_text(self):
        with self.assertRaises(EditorError): compare_transcript('!!!', 'Hello')
    def test_speed_duration_conflict(self):
        with self.assertRaises(EditorError): build_plan(EditSpec(operation='speed',speed=2,target_seconds=4),4,self.limits)
    def test_immutable_runtime_pin(self):
        self.assertEqual(len(RUNTIME_SHA256),64); self.assertEqual(len(MODEL_REVISION),40)


class MediaAndJobTests(unittest.TestCase):
    def setUp(self):
        self.tmp=tempfile.TemporaryDirectory(); self.root=Path(self.tmp.name)
        self.wav=make_test_wav(self.root/'voice.wav')
        self.limits=Limits(); self.editors=[]
        FakeTransport.calls=0; FakeTransport.fail=False
    def tearDown(self):
        for e in self.editors: e.close()
        self.tmp.cleanup()
    def editor(self, **kwargs):
        e=VoiceEditor(self.root/'worker', self.root/'binary', self.root/'models',
                      memory_saver=True,provenance={'test':'fixture'},transport_factory=FakeTransport,**kwargs)
        self.editors.append(e); return e
    def test_decode_valid_audio(self):
        data=self.wav.read_bytes()
        self.assertEqual(decode_response({'audio':base64.b64encode(data).decode()},self.limits),data)
    def test_missing_audio_rejected(self):
        with self.assertRaises(EditorError): decode_response({},self.limits)
    def test_bad_base64_rejected(self):
        with self.assertRaises(EditorError): decode_response({'audio':'$$$'},self.limits)
    def test_non_wav_rejected(self):
        with self.assertRaises(EditorError): decode_response({'audio':base64.b64encode(b'x'*100).decode()},self.limits)
    def test_zero_byte_input(self):
        p=self.root/'empty.wav';p.touch()
        with self.assertRaises(EditorError): probe_audio(p,self.limits)
    def test_corrupt_input(self):
        p=self.root/'bad.wav';p.write_bytes(b'not audio')
        with self.assertRaises(EditorError): probe_audio(p,self.limits)
    def test_silent_input(self):
        p=make_test_wav(self.root/'silence.wav',amplitude=0)
        with self.assertRaises(EditorError): prepare_audio(p,self.root/'prepared.wav',0,None,self.limits)
    def test_stereo_downmix(self):
        p=make_test_wav(self.root/'stereo.wav',channels=2)
        result=prepare_audio(p,self.root/'prepared.wav',0,None,self.limits)
        self.assertTrue(any('downmixed' in s for s in result['warnings']))
    def test_partial_clip_explicit_warning(self):
        result=prepare_audio(self.wav,self.root/'prepared.wav',.25,.8,self.limits)
        self.assertTrue(any('NOT spliced' in s for s in result['warnings']))
    def test_oversized_duration_not_silently_trimmed(self):
        small=dataclasses.replace(self.limits,max_input_seconds=.6)
        with self.assertRaises(EditorError): prepare_audio(self.wav,self.root/'prepared.wav',0,None,small)
    def test_wrong_channel(self):
        with self.assertRaises(EditorError): prepare_audio(self.wav,self.root/'prepared.wav',0,None,self.limits,'right')
    def test_output_checks(self):
        self.assertEqual(validate_output(self.wav,1,self.limits)['technical_checks'],'passed')
    def test_wrong_duration(self):
        with self.assertRaises(EditorError): validate_output(self.wav,2,self.limits)
    def test_wrong_sample_rate(self):
        p=make_test_wav(self.root/'rate.wav',rate=16000)
        with self.assertRaises(EditorError): validate_output(p,1,self.limits)
    def test_generated_not_approved(self):
        e=self.editor(); r=e.edit(self.wav,EditSpec(operation='emotion'),rights_confirmed=True)
        self.assertEqual(r['status'],'generated'); self.assertEqual(r['human_review'],'pending')
        self.assertTrue(Path(r['output_path']).is_file())
    def test_idempotent_repeat(self):
        e=self.editor(); spec=EditSpec(operation='emotion')
        a=e.edit(self.wav,spec,rights_confirmed=True); b=e.edit(self.wav,spec,rights_confirmed=True)
        self.assertEqual(a['job_id'],b['job_id']); self.assertTrue(b['cache_hit']);self.assertEqual(FakeTransport.calls,1)
    def test_cache_corruption(self):
        e=self.editor();spec=EditSpec(operation='emotion');a=e.edit(self.wav,spec,rights_confirmed=True)
        Path(a['output_path']).write_bytes(b'changed')
        with self.assertRaises(EditorError):e.edit(self.wav,spec,rights_confirmed=True)
    def test_rights_required(self):
        with self.assertRaises(EditorError): self.editor().edit(self.wav,EditSpec(operation='emotion'))
    def test_commercial_gate(self):
        with self.assertRaises(EditorError): self.editor(commercial=True)
    def test_exclusive_worker_lock(self):
        e=self.editor()
        with self.assertRaises(EditorError):self.editor()
    def test_failed_requires_explicit_retry(self):
        e=self.editor(); spec=EditSpec(operation='emotion');FakeTransport.fail=True
        with self.assertRaises(EditorError):e.edit(self.wav,spec,rights_confirmed=True)
        FakeTransport.fail=False
        with self.assertRaises(EditorError):e.edit(self.wav,spec,rights_confirmed=True)
        r=e.edit(self.wav,spec,rights_confirmed=True,retry_failed=True)
        self.assertEqual(r['attempts'],2);self.assertEqual(r['status'],'generated')
    def test_no_output_after_failure(self):
        e=self.editor();FakeTransport.fail=True
        with self.assertRaises(EditorError): e.edit(self.wav,EditSpec(operation='emotion'),rights_confirmed=True)
        self.assertFalse(list((e.root/'jobs').rglob('output.wav')))
    def test_word_edit_without_qa_blocked(self):
        e=self.editor()
        spec=EditSpec(operation='replace_words',source_transcript='Say yes',old_words='yes',new_words='no')
        with self.assertRaises(EditorError):e.edit(self.wav,spec,rights_confirmed=True)
    def test_approval_explicit(self):
        e=self.editor();r=e.edit(self.wav,EditSpec(operation='emotion'),rights_confirmed=True)
        m=e.review(r['job_id'],approved=True,notes='Test fixture only')
        self.assertEqual(m['human_review'],'approved')
    def test_review_path_traversal(self):
        e=self.editor()
        with self.assertRaises(EditorError):e.review('../bad',approved=True)
    def test_asr_discrepancy_flags_review(self):
        counter=[0]
        def asr(path):
            counter[0]+=1
            return {'text':'Hello world' if counter[0]==1 else 'Goodbye world'}
        e=self.editor(asr=asr,asr_fingerprint='test')
        r=e.edit(self.wav,EditSpec(operation='emotion'),rights_confirmed=True)
        self.assertEqual(r['semantic_validation'],'review_required')
    def test_stale_job_recovery(self):
        e=self.editor();jid=uuid.uuid4().hex
        e.db.execute('INSERT INTO jobs VALUES (?,?,?,?)',(jid,'stale','running',utc_now()));e.db.commit();e.close()
        reopened=self.editor()
        row=reopened.db.execute('SELECT status FROM jobs WHERE job_id=?',(jid,)).fetchone()
        self.assertEqual(row[0],'interrupted')
    def test_sql_injection_like_prompt_is_data(self):
        e=self.editor();r=e.edit(self.wav,EditSpec(instruction="'; DROP TABLE jobs; --"),rights_confirmed=True)
        self.assertEqual(e.db.execute('SELECT COUNT(*) FROM jobs').fetchone()[0],1)
    def test_job_manifests_do_not_store_base64_response(self):
        e=self.editor();r=e.edit(self.wav,EditSpec(operation='emotion'),rights_confirmed=True)
        m=json.loads((Path(r['job_dir'])/'manifest.json').read_text())
        self.assertNotIn('audio',m);self.assertNotIn(base64.b64encode(self.wav.read_bytes()).decode(),json.dumps(m))


class ArtifactTests(unittest.TestCase):
    def test_parse_lfs_pointer(self):
        digest='a'*64
        self.assertEqual(parse_lfs_pointer(f'version https://git-lfs.github.com/spec/v1\noid sha256:{digest}\nsize 123\n'),(digest,123))
    def test_bad_pointer(self):
        with self.assertRaises(EditorError):parse_lfs_pointer('some html error')
    def test_archive_traversal(self):
        with tempfile.TemporaryDirectory() as tmp:
            root=Path(tmp);p=root/'archive.tar.gz'
            with tarfile.open(p,'w:gz') as t:
                m=tarfile.TarInfo('../escape');m.size=1;t.addfile(m,io.BytesIO(b'x'))
            with self.assertRaises(EditorError):safe_extract(p,root/'out')
            self.assertFalse((root/'escape').exists())
    def test_archive_symlink(self):
        with tempfile.TemporaryDirectory() as tmp:
            root=Path(tmp);p=root/'archive.tar.gz'
            with tarfile.open(p,'w:gz') as t:
                m=tarfile.TarInfo('link');m.type=tarfile.SYMTYPE;m.linkname='/etc';t.addfile(m)
            with self.assertRaises(EditorError):safe_extract(p,root/'out')
    def test_safe_archive(self):
        with tempfile.TemporaryDirectory() as tmp:
            root=Path(tmp);p=root/'archive.tar.gz'
            with tarfile.open(p,'w:gz') as t:
                m=tarfile.TarInfo('file');m.size=1;t.addfile(m,io.BytesIO(b'x'))
            safe_extract(p,root/'out');self.assertEqual((root/'out'/'file').read_bytes(),b'x')
    def test_canonical_hash_stable(self):
        self.assertEqual(canonical_hash({'b':2,'a':1}),canonical_hash({'a':1,'b':2}))
    def test_local_server_hard_stop(self):
        p=subprocess.Popen([sys.executable,'-c','import time;time.sleep(60)'],start_new_session=True)
        try:
            ticks=process_start_ticks(p.pid);kill_owned_group(p.pid,ticks);p.wait(timeout=6)
            self.assertIsNotNone(p.returncode)
        finally:
            if p.poll() is None:p.kill();p.wait()
    def test_pid_reuse_guard(self):
        p=subprocess.Popen([sys.executable,'-c','import time;time.sleep(60)'],start_new_session=True)
        try:
            kill_owned_group(p.pid,'not-the-start-time');self.assertIsNone(p.poll())
        finally:p.kill();p.wait()


FAKE_HTTP_SCRIPT = r'''#!/usr/bin/env python3
import argparse,base64,json,time
from http.server import BaseHTTPRequestHandler,HTTPServer
p=argparse.ArgumentParser();p.add_argument('--config');p.add_argument('--no-ui',action='store_true');a=p.parse_args()
c=json.load(open(a.config));model_id=c['models'][0]['id']
class H(BaseHTTPRequestHandler):
 def log_message(self,*a):pass
 def reply(self,obj,status=200):
  data=json.dumps(obj).encode();self.send_response(status);self.send_header('Content-Type','application/json');self.send_header('Content-Length',str(len(data)));self.end_headers();self.wfile.write(data)
 def do_GET(self):self.reply({'data':[{'id':model_id}]} if self.path=='/v1/models' else {'status':'ok'})
 def do_POST(self):
  r=json.loads(self.rfile.read(int(self.headers['Content-Length'])))['request']
  if r['text']=='HANG':time.sleep(60)
  if r['text']=='ERROR':self.reply({'error':'out of memory'},500);return
  if r['text']=='BAD':self.reply({'foo':'bar'});return
  self.reply({'audio':base64.b64encode(open(r['audio'],'rb').read()).decode()})
HTTPServer((c['host'],c['port']),H).serve_forever()
'''

class TransportIntegrationTests(unittest.TestCase):
    def setUp(self):
        self.tmp=tempfile.TemporaryDirectory();self.root=Path(self.tmp.name)
        self.binary=self.root/'server';self.binary.write_text(FAKE_HTTP_SCRIPT);self.binary.chmod(0o755)
        self.models=self.root/'models';self.models.mkdir()
        for name in (PROFILE_FILES['base_f32'][1],*SHARED_FILES,*SIDECARS):
            p=self.models/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text('fixture')
        self.wav=make_test_wav(self.root/'voice.wav')
        self.server=LocalAuKServer(self.binary,self.models,self.root/'runtime',
                                  dataclasses.replace(Limits(),startup_timeout=5,inference_timeout=.8),True)
    def tearDown(self):self.server.stop();self.tmp.cleanup()
    def plan(self,instruction='ok'):
        return {'instruction':instruction,'seed':42,'target_seconds':1}
    def test_real_http_request_shape(self):
        r,t=self.server.run('base_f32',self.wav,self.plan())
        self.assertEqual(decode_response(r,Limits()),self.wav.read_bytes())
        self.assertTrue(t['first_request_after_start'])
    def test_warm_request_marker(self):
        self.server.run('base_f32',self.wav,self.plan())
        r,t=self.server.run('base_f32',self.wav,self.plan())
        self.assertFalse(t['first_request_after_start'])
    def test_http_oom_stops_process(self):
        with self.assertRaises(EditorError) as cm:self.server.run('base_f32',self.wav,self.plan('ERROR'))
        self.assertEqual(cm.exception.code,'out_of_memory');self.assertIsNone(self.server.process)
    def test_watchdog_kills_hanging_http_server(self):
        with self.assertRaises(EditorError):self.server.run('base_f32',self.wav,self.plan('HANG'))
        self.assertIsNone(self.server.process);self.assertFalse(self.server.owner_file.exists())
    def test_bad_payload_rejected_after_http_success(self):
        r,_=self.server.run('base_f32',self.wav,self.plan('BAD'))
        with self.assertRaises(EditorError):decode_response(r,Limits())


def run_self_tests():
    suite=unittest.TestSuite()
    for cls in (PlannerTests,MediaAndJobTests,ArtifactTests,TransportIntegrationTests):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    result=unittest.TextTestRunner(verbosity=1).run(suite)
    if not result.wasSuccessful(): raise RuntimeError('Self-tests failed. Do not run inference until fixed.')
    return {'tests_run':result.testsRun,'failures':len(result.failures),'errors':len(result.errors),
            'scope':'CPU media processing + synthetic/mocked inference plumbing. No AuK or ASR inference.'}


In [ ]:
# @title Run all 64 implementation tests
SELF_TEST_RESULT = run_self_tests()
print(SELF_TEST_RESULT)


## 3. Prepare the GPU runtime and pinned weights
The prebuilt runtime is pinned to `v0.9.0` and its published archive SHA-256. AuK weights and tokenizer/config sidecars are pinned to one full Hugging Face commit. Weight files are checked against the SHA-256 pointers in that commit. Downloads can resume after a connection failure; mismatched final hashes are rejected.

The optional source-build path verifies the release's observed commit prefix and records the resolved full commit. Promote that full commit into your deployment lock after acceptance. Source compilation and downloading weights are setup operations, never per-edit work.

In [ ]:
# @title Preflight, runtime setup, and model download
LIMITS = Limits()
GPU = gpu_details()
print(GPU)
if GPU["total_mib"] < 14000:
    raise EditorError("insufficient_vram", "This notebook's profiles require a GPU with roughly 16 GiB or more total VRAM.")
if shutil.disk_usage(ROOT).free < 4 * 2**30:
    raise EditorError("disk_space", "At least 4 GiB working space is required before model downloads.")
MEMORY_SAVER = MEMORY_MODE == "save_memory" or (MEMORY_MODE == "auto" and GPU["free_mib"] < 22 * 1024)
BINARY, RUNTIME_LOCK = setup_runtime(ROOT, mode=RUNTIME_MODE, gpu=GPU)
DOWNLOAD_PROFILES = sorted(set([PROFILE] + (["base_f32", "flash_f16"] if DOWNLOAD_BASE_AND_FLASH else [])))
MODEL_DIR, MODEL_LOCK = setup_models(ROOT, DOWNLOAD_PROFILES)
ENVIRONMENT = {
    "python": sys.version, "gpu": GPU,
    "nvidia_smi": command(["nvidia-smi"], timeout=10),
    "ffmpeg": command(["ffmpeg", "-version"]).splitlines()[0],
    "source_bundle_sha256": "6ad51f729eff36d0b96f159161cb81a75257eaed0f278fb39fa89b10a0dfcf1c",
}
atomic_json(ROOT / "environment.json", ENVIRONMENT)
print("Prepared profiles:", DOWNLOAD_PROFILES)
print("Memory-saving mode:", MEMORY_SAVER)
print("Server is not yet warmed. First generation is marked separately from later requests.")


## 4. Optional speech QA and worker startup
Set `ENABLE_ASR_QA=True` in section 1 to enable CPU transcription checks. It installs `faster-whisper==1.2.1` in a separate environment, uses a pinned Whisper-small model, then saves the complete resolved dependency list. Those transitive package versions are frozen **after first installation**; reuse that saved lock in deployment.

ASR is an imperfect content-checking heuristic. It cannot certify identity, naturalness, emotion, or that an unchanged transcription reflects unchanged audio. Text discrepancies remain flagged for review, not silently regenerated. Word replacement is blocked without QA unless you explicitly allow an unchecked evaluation output. Free-form prompts receive a text check only when an expected transcript is supplied.

In [ ]:
# @title Optional CPU ASR setup and reusable worker
ASR_CALLBACK, ASR_FINGERPRINT = None, "disabled"
if ENABLE_ASR_QA:
    ASR_CALLBACK, ASR_FINGERPRINT = setup_asr(ROOT, LIMITS, ASR_LANGUAGE)
    print("CPU speech QA is enabled.")
else:
    print("Speech QA is disabled. Output content must be reviewed manually.")
if "EDITOR" in globals() and EDITOR is not None:
    EDITOR.close()
PROVENANCE = {
    "runtime": RUNTIME_LOCK,
    "model": MODEL_LOCK,
    "environment": ENVIRONMENT,
}
EDITOR = VoiceEditor(ROOT / "workspace", BINARY, MODEL_DIR,
    memory_saver=MEMORY_SAVER, provenance=PROVENANCE, limits=LIMITS,
    commercial=COMMERCIAL_USE, commercial_license_confirmed=COMMERCIAL_LICENSE_CONFIRMED,
    asr=ASR_CALLBACK, asr_fingerprint=ASR_FINGERPRINT)
import atexit
atexit.register(EDITOR.close)
print("Worker ready for one local job at a time. GPU inference readiness still needs a successful first generation.")


## 5. Upload one recording
Accepted extensions: WAV, MP3, FLAC, M4A, MP4, MOV, OGG, Opus, WebM. Maximum upload: 64 MiB. Exactly one audio stream is required. Video input returns a voice-track WAV, not an edited video.

For a long recording, select a range in section 6. There is no hidden first-15-seconds truncation. Uploads and job files remain private within your Colab VM, not a public Space or tunnel. Colab storage is temporary; export the outputs you need before the runtime is deleted.

In [ ]:
# @title Upload, or supply an existing local file
LOCAL_INPUT_PATH = "/content/exampleinput (1).mp3"  # @param {type:"string"}

if LOCAL_INPUT_PATH.strip():
    SOURCE_FILE = Path(LOCAL_INPUT_PATH).expanduser().resolve()
    INPUT_INFO = probe_audio(SOURCE_FILE, LIMITS)
else:
    from google.colab import files as colab_files
    upload_dir = ROOT / "uploads" / uuid.uuid4().hex
    upload_dir.mkdir(parents=True, mode=0o700)
    previous_dir = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = colab_files.upload()
    finally:
        os.chdir(previous_dir)
    if len(uploaded) != 1:
        shutil.rmtree(upload_dir)
        raise EditorError("one_file_required", "Upload exactly one recording.")
    original_name, data = next(iter(uploaded.items()))
    if not data or len(data) > LIMITS.max_upload_bytes:
        shutil.rmtree(upload_dir)
        raise EditorError("upload_size", "Provide a nonempty recording of at most 64 MiB.")
    suffix = Path(original_name).suffix.lower()
    SOURCE_FILE = upload_dir / ("source" + suffix)
    atomic_bytes(SOURCE_FILE, data)
    # Remove the uploader's extra copy; only the normalized filename is retained.
    for candidate in upload_dir.iterdir():
        if candidate != SOURCE_FILE and candidate.is_file(): candidate.unlink()
    del uploaded, data
    INPUT_INFO = probe_audio(SOURCE_FILE, LIMITS)
print("Input:", SOURCE_FILE.name)
print(INPUT_INFO)


## 6. Edit the voice
Choose a preset or `custom`. Set **RUN_EDIT** and **RIGHTS_CONFIRMED**, then run this cell. Repeating identical settings returns the verified cached result rather than consuming another generation. Change the seed for another take. A failed attempt requires **RETRY_FAILED=True**.

`END_SECONDS=0` means the end of the file. `TARGET_SECONDS=0` preserves the selected clip's duration, except `speed`, which computes duration from the speed factor. For content-changing custom instructions, explicitly choose the target duration and supply `EXPECTED_TRANSCRIPT` when possible.

`replace_words` requires a checked source transcript and a phrase appearing exactly once. It regenerates the **whole selected clip**; it does not claim word-aligned localized splicing. The instruction shown is the requested change, never a claimed transcription of the result.

In [ ]:
# @title Generate an edit
RUN_EDIT = False  # @param {type:"boolean"}
RIGHTS_CONFIRMED = False  # @param {type:"boolean"}
OPERATION = "emotion"  # @param ["custom", "emotion", "voice", "whisper", "enhance", "denoise", "dereverb", "speed", "replace_words"]
INSTRUCTION = "Make the voice sound more excited. Keep the words unchanged."  # @param {type:"string"}
EMOTION = "excited"  # @param ["happy", "angry", "sad", "fearful", "surprised", "disgusted", "calm", "excited"]
VOICE_DESCRIPTION = "a warmer, deeper voice"  # @param {type:"string"}
START_SECONDS = 0.0  # @param {type:"number"}
END_SECONDS = 0.0  # @param {type:"number"}
CHANNEL = "mix"  # @param ["mix", "left", "right"]
SPEED = 1.0  # @param {type:"number"}
TARGET_SECONDS = 0.0  # @param {type:"number"}
OLD_WORDS = ""  # @param {type:"string"}
NEW_WORDS = ""  # @param {type:"string"}
SOURCE_TRANSCRIPT = ""  # @param {type:"string"}
EXPECTED_TRANSCRIPT = ""  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}
RETRY_FAILED = False  # @param {type:"boolean"}
ALLOW_UNCHECKED_WORD_EDITS = False  # @param {type:"boolean"}

RESULT = None  # Clear stale display state before a new attempt.
CURRENT_SPEC = EditSpec(operation=OPERATION, instruction=INSTRUCTION, emotion=EMOTION,
    voice_description=VOICE_DESCRIPTION, old_words=OLD_WORDS, new_words=NEW_WORDS,
    speed=SPEED, target_seconds=None if TARGET_SECONDS == 0 else TARGET_SECONDS,
    source_transcript=SOURCE_TRANSCRIPT, expected_transcript=EXPECTED_TRANSCRIPT, seed=SEED)
CURRENT_OPTIONS = dict(profile=PROFILE, start_seconds=START_SECONDS,
    end_seconds=None if END_SECONDS == 0 else END_SECONDS, channel=CHANNEL,
    rights_confirmed=RIGHTS_CONFIRMED, retry_failed=RETRY_FAILED,
    allow_unchecked_word_edits=ALLOW_UNCHECKED_WORD_EDITS)
if RUN_EDIT:
    RESULT = EDITOR.edit(SOURCE_FILE, CURRENT_SPEC, **CURRENT_OPTIONS)
    print("Job:", RESULT["job_id"], "| profile:", RESULT["profile_used"], "| cache hit:", RESULT["cache_hit"])
    print("Requested instruction:", RESULT["plan"]["instruction"])
    print("Timing:", RESULT["timing"])
    print("Content-check status:", RESULT["semantic_validation"])
    for warning in RESULT["warnings"]: print("Warning:", warning)
    from IPython.display import Audio, display
    print("Before (selected clip):")
    display(Audio(filename=str(Path(RESULT["job_dir"]) / "input.wav")))
    print("After - unapproved draft:")
    display(Audio(filename=RESULT["output_path"]))
    if "output_asr" in RESULT["quality"]:
        print("ASR-estimated output text:", RESULT["quality"]["output_asr"]["text"])
else:
    print("No edit run. Set RUN_EDIT and RIGHTS_CONFIRMED when ready, then run this cell.")


## 7. Review and download
Listen for correct words, identity consistency, requested delivery, unwanted noise, seam artifacts, and truncated endings. No result is automatically human-approved, even when every technical check passes.

Downloads are labeled `draft`, `approved`, or `rejected` from the saved review state. A user download is not a publication or a commercial clearance. The optional package includes private source audio, prompts, and transcripts; do not share it without reviewing its contents.

In [ ]:
# @title Save a review and/or download the last output
SAVE_REVIEW = False  # @param {type:"boolean"}
APPROVE_AFTER_LISTENING = False  # @param {type:"boolean"}
REVIEW_NOTES = ""  # @param {type:"string"}
DOWNLOAD_OUTPUT = False  # @param {type:"boolean"}
DOWNLOAD_FULL_JOB_PACKAGE = False  # @param {type:"boolean"}

if SAVE_REVIEW or DOWNLOAD_OUTPUT or DOWNLOAD_FULL_JOB_PACKAGE:
    if RESULT is None:
        raise RuntimeError("Generate a successful result in section 6 first.")
    job_dir = Path(RESULT["job_dir"])
    if SAVE_REVIEW:
        EDITOR.review(RESULT["job_id"], approved=APPROVE_AFTER_LISTENING, notes=REVIEW_NOTES)
    saved = json.loads((job_dir / "manifest.json").read_text())
    output_path = job_dir / "output.wav"
    if sha256_file(output_path) != saved["output_sha256"]:
        raise RuntimeError("Output integrity check failed. Download blocked.")
    status = saved["human_review"] if saved["human_review"] in ("approved", "rejected") else "draft"
    from google.colab import files as colab_files
    exports = ROOT / "exports"; exports.mkdir(exist_ok=True)
    if DOWNLOAD_OUTPUT:
        named = exports / f"voice-edit-{saved['job_id'][:8]}-{status}.wav"
        shutil.copy2(output_path, named)
        colab_files.download(str(named))
    if DOWNLOAD_FULL_JOB_PACKAGE:
        import zipfile
        package = exports / f"voice-edit-{saved['job_id'][:8]}-{status}.zip"
        with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
            for filename in ("input.wav", "output.wav", "manifest.json"):
                z.write(job_dir / filename, filename)
        colab_files.download(str(package))
    print("Saved human review:", saved["human_review"])
else:
    print("No review saved and no download requested.")


## 8. Optional Base/Flash cold/warm comparison
Uses your current uploaded clip and edit settings. **Each row is a real new generation**, not a cache lookup. Two runs per profile are the default: first after a restart, then a repeated request. In memory-saving mode the second request still reloads stages, so do not assume it represents fully resident inference.

Choose profiles downloaded in section 3. The cost estimate uses your supplied hourly rate multiplied by measured end-to-end job time. It excludes installation, model downloads, idle periods between jobs, persistent storage, and broader infrastructure; it is not a fully allocated production unit cost.

In [ ]:
# @title Run a measured comparison (off by default)
RUN_BENCHMARK = False  # @param {type:"boolean"}
BENCHMARK_PROFILES = "base_f32,flash_f16"  # @param {type:"string"}
REPEATS_PER_PROFILE = 2  # @param {type:"integer"}
GPU_HOURLY_USD = 0.0  # @param {type:"number"}

if RUN_BENCHMARK:
    if not RIGHTS_CONFIRMED: raise RuntimeError("Confirm recording rights first.")
    if not 2 <= REPEATS_PER_PROFILE <= 5: raise ValueError("Use 2-5 repetitions per profile.")
    hourly = require_finite(GPU_HOURLY_USD, "GPU_HOURLY_USD", 0, 100)
    selected = [p.strip() for p in BENCHMARK_PROFILES.split(",") if p.strip()]
    if not selected or any(p not in DOWNLOAD_PROFILES for p in selected):
        raise ValueError("Download all selected benchmark profiles in section 3 first.")
    run_id = uuid.uuid4().hex
    benchmark = []
    for p in selected:
        EDITOR.transport.stop()
        for repeat in range(REPEATS_PER_PROFILE):
            options = {**CURRENT_OPTIONS, "profile": p, "retry_failed": False,
                       "cache_namespace": f"benchmark-{run_id}-{p}-{repeat}"}
            try:
                r = EDITOR.edit(SOURCE_FILE, CURRENT_SPEC, **options)
                t = r["timing"]
                row = {"profile":p, "run":repeat+1, "job_id":r["job_id"],
                       "status":r["status"], "first_after_start":t["first_request_after_start"],
                       "inference_seconds":t["inference_seconds"], "total_seconds":t["total_seconds"],
                       "rtf":t["inference_real_time_factor"], "memory_saver":t["memory_saver"],
                       "text_check":r["semantic_validation"], "human_review":r["human_review"],
                       "active_job_cost_usd":round(t["total_seconds"]*hourly/3600,5) if hourly else None}
            except EditorError as exc:
                row={"profile":p,"run":repeat+1,"status":"failed","error_code":exc.code}
            benchmark.append(row); print(json.dumps(row))
    atomic_json(ROOT / "benchmark.json", {"run_id":run_id,"hourly_usd":hourly,"rows":benchmark})
    print("Saved measured results to", ROOT / "benchmark.json")
else:
    print("Benchmark is off; no extra inference is run.")


## 9. Diagnostics and shutdown
Diagnostics below exclude audio, prompt text, transcripts, and raw server logs. Raw logs remain private in the working directory; inspect them locally after an error. A timeout or cancellation stops the owned inference process before another job can run. The next request can start a new server, but failed jobs require an explicit retry.

In [ ]:
# @title Export content-free diagnostics or stop the worker
EXPORT_DIAGNOSTICS = False  # @param {type:"boolean"}
STOP_WORKER = False  # @param {type:"boolean"}

if EXPORT_DIAGNOSTICS:
    records=[]
    for job_id, status, updated in EDITOR.db.execute("SELECT job_id,status,updated_at FROM jobs ORDER BY updated_at DESC LIMIT 100").fetchall():
        p=EDITOR.root / "jobs" / job_id / "manifest.json"
        m=json.loads(p.read_text()) if p.is_file() else {}
        records.append({"job_id":job_id,"status":status,"updated_at":updated,
                        "profile":m.get("profile_used"),"error_code":m.get("error_code"),
                        "timing":m.get("timing"),"attempts":m.get("attempts"),
                        "semantic_validation":m.get("semantic_validation")})
    diagnostic=ROOT / "diagnostics.json"
    atomic_json(diagnostic,{"self_tests":SELF_TEST_RESULT,"environment":ENVIRONMENT,
                            "runtime":RUNTIME_LOCK,"model":MODEL_LOCK,"jobs":records})
    from google.colab import files as colab_files
    colab_files.download(str(diagnostic))
if STOP_WORKER:
    EDITOR.close()
    print("Stopped. Rerun section 4 to create another worker before editing.")
if not EXPORT_DIAGNOSTICS and not STOP_WORKER:
    print("Worker remains available in this Colab session.")


Worker remains available in this Colab session.


## GPU acceptance checklist and production handoff

The notebook intentionally does **not** label a result `successful_edit` or auto-approve it. `generated` means the WAV passed technical checks; inspect `semantic_validation` and `human_review` separately.

Before customer deployment, run Base and Flash on representative, authorized recordings: short/long clips within the limits, accents/languages you intend to support, quiet/loud/clipped speech, multiple speakers, and dialogue with background sound. Include untouched speech outside the requested change in your listening rubric. Use measured p50/p95 latency, peak VRAM, failure rate, content correctness, and human approval rate to select per-operation profiles. The small benchmark is not a statistically adequate latency or quality study.

Test each intended GPU with resident and memory-saving mode, interrupt an actual GPU job, and rerun the same idempotency key. Confirm the original process releases VRAM, no partial output is surfaced, and a completed request is not charged twice by the billing integration you add. The notebook itself performs **no billing** and makes no distributed exactly-once guarantee.

**Production adapter boundary:** upload/ownership validation → durable queue → `VoiceEditor.edit`-equivalent worker → output validation → private object storage → status endpoint. Do not expose the local audio.cpp server directly: it accepts server-side paths and does not implement your application's authentication, tenant isolation, authorization, or abuse controls. Keep the model/backend replaceable.

The current implementation exports a selected clip only. Long-recording edits need an additional aligned segmentation/splice layer with continuity and video timing tests. Speaker isolation, lyric editing, and arbitrary sound editing are not product promises here. Single-user file locks and SQLite are not a multi-host job queue.

### Artifact validation performed
- 64 tests executed successfully with actual FFmpeg/FFprobe and local HTTP fixtures.
- All notebook code cells are syntax-checked after IPython transformation.
- Notebook structure validated with `nbformat`.
- **Not executed here:** Colab system installation, model downloads, T4/CUDA inference, source compilation, optional Whisper inference, perceptual listening, and browser upload/download controls.
- To finish validation: open in a T4 Colab runtime, run sections 1-5, edit a short authorized clip in section 6, listen and inspect QA, then run section 8 on both profiles. Read failure diagnostics rather than assuming a completed setup establishes model quality.

### Source notes
Source file: your uploaded `tencent.ipynb`, whose local `/v1/tasks/run` route and F32 Base / Q8 conditioner / F32 VAE configuration were retained as the baseline.

External implementation references checked October 1, 2026:
- [audio.cpp AuK options](https://raw.githubusercontent.com/0xShug0/audio.cpp/main/docs/community_models/auk.md): variant-specific schedules, component names, mono 24 kHz output, memory-saving behavior.
- [audio.cpp v0.9.0 release](https://github.com/0xShug0/audio.cpp/releases/tag/v0.9.0): pinned runtime asset and published archive digest.
- [audio.cpp release workflow](https://raw.githubusercontent.com/0xShug0/audio.cpp/main/.github/workflows/release.yml): Colab prebuilt targets compute capability 7.5.
- [audio.cpp server contract](https://raw.githubusercontent.com/0xShug0/audio.cpp/main/app/server/README.md): local server setup, health/models endpoints, generic task request shape.
- [AuK GGUF pinned revision](https://huggingface.co/audio-cpp/AuK-Base-and-Flash-GGUF/tree/2814a16a49b822a5709e11a78cfd7c1662f19b61): weight and sidecar provenance. Component/quantized validation is narrower than full task-quality validation.
- [Tencent cookbook](https://raw.githubusercontent.com/Tencent-Hunyuan/AuK/main/docs/COOKBOOK.md): instruction categories and explicit duration for content/speed edits. Preset instructions in this notebook are implementation choices, not a benchmarked prompt guarantee.
- [Qwen2.5-Omni-3B license](https://huggingface.co/Qwen/Qwen2.5-Omni-3B/raw/main/LICENSE): research/evaluation limitation and separate commercial-license requirement.
- [faster-whisper documentation](https://github.com/SYSTRAN/faster-whisper): CPU INT8 transcription and word timestamps.
- [Whisper-small pinned revision](https://huggingface.co/Systran/faster-whisper-small/tree/536b0662742c02347bc0e980a01041f333bce120): optional QA model.

No model weights are embedded in or redistributed with this notebook. Downloads remain subject to their respective licenses. Private job manifests contain instructions, source hashes, and transcripts when QA is enabled; apply your retention policy before production use.